In [2]:
# ================================================================
# TEST — READ ONE API ORDER
# ================================================================

TEST_PATH = (
    "s3://aws-ecommerce-s3/"
    "ecommerce/api_raw/orders/order_50974.json"
)

print("=" * 80)
print("TESTING SINGLE API ORDER")
print("=" * 80)

test_order = (
    spark.read
    .option("multiLine", True)
    .json(TEST_PATH)
)

test_order.printSchema()

test_order.show(
    truncate=False
)

print("Single JSON file read successfully.")

AnalysisException: Path does not exist: s3://aws-ecommerce-s3/ecommerce/api_raw/orders/order_50974.json


In [3]:
# ================================================================
# DIAGNOSTIC — COMPARE BOTO3 AND SPARK ACCESS
# ================================================================

import boto3

BUCKET = "aws-ecommerce-s3"
KEY = "ecommerce/api_raw/orders/order_50974.json"
PATH = f"s3://{BUCKET}/{KEY}"

print("=" * 80)
print("1. BOTO3 HEAD_OBJECT")
print("=" * 80)

s3 = boto3.client("s3")

try:
    obj = s3.head_object(
        Bucket=BUCKET,
        Key=KEY
    )

    print("BOTO3: OBJECT EXISTS")
    print("Size:", obj["ContentLength"])
    print("LastModified:", obj["LastModified"])
    print("ETag:", obj["ETag"])

except Exception as e:
    print("BOTO3 ERROR:")
    print(type(e).__name__)
    print(e)


print("\n" + "=" * 80)
print("2. SPARK FILESYSTEM CHECK")
print("=" * 80)

try:
    jvm = spark._jvm

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    fs = jvm.org.apache.hadoop.fs.FileSystem.get(
        jvm.org.apache.hadoop.fs.Path(PATH).toUri(),
        hadoop_conf
    )

    path_obj = jvm.org.apache.hadoop.fs.Path(PATH)

    exists = fs.exists(path_obj)

    print("Spark Hadoop FS exists:", exists)

except Exception as e:
    print("SPARK FILESYSTEM ERROR:")
    print(type(e).__name__)
    print(e)


print("\n" + "=" * 80)
print("3. SPARK READ TEST")
print("=" * 80)

try:

    test_df = (
        spark.read
        .option("multiLine", True)
        .json(PATH)
    )

    print("Spark successfully created DataFrame.")

    test_df.printSchema()

    test_df.show(
        1,
        truncate=False
    )

except Exception as e:
    print("SPARK READ ERROR:")
    print(type(e).__name__)
    print(e)

1. BOTO3 HEAD_OBJECT
BOTO3 ERROR:
ClientError
An error occurred (403) when calling the HeadObject operation: Forbidden

2. SPARK FILESYSTEM CHECK
Spark Hadoop FS exists: False

3. SPARK READ TEST
SPARK READ ERROR:
AnalysisException
Path does not exist: s3://aws-ecommerce-s3/ecommerce/api_raw/orders/order_50974.json


In [5]:
# ================================================================
# PART 1 — DATA UNDERSTANDING
# ================================================================
#
# Purpose:
#   1. Read all historical source tables from their actual S3 paths.
#   2. Read all incremental API orders from the JSON S3 prefix.
#   3. Display schema for every source.
#   4. Display 10 sample records.
#   5. Count records and columns.
#   6. Identify business/primary keys.
#   7. Document foreign-key relationships.
#   8. Identify nullable columns from the Spark schema.
#
# Important:
#   The API JSON files are NOT concatenated.
#   Spark reads the entire S3 prefix as one logical DataFrame.
#
# ================================================================


from pyspark.sql import functions as F


# ================================================================
# 1. CONFIGURATION
# ================================================================

BUCKET = "s3://aws-ecommerce-s3"

HISTORICAL_BASE_PATH = f"{BUCKET}/ecommerce/raw"
API_ORDERS_PATH = f"{BUCKET}/ecommerce/api_raw/orders"


# ================================================================
# 2. SOURCE TABLE PATHS
# ================================================================
#
# Historical files are stored inside individual folders.
#
# Example:
#   ecommerce/raw/customers/customers.csv
#
# ================================================================

historical_paths = {
    "customers":
        f"{HISTORICAL_BASE_PATH}/customers/customers.csv",

    "categories":
        f"{HISTORICAL_BASE_PATH}/categories/categories.csv",

    "products":
        f"{HISTORICAL_BASE_PATH}/products/products.csv",

    "departments":
        f"{HISTORICAL_BASE_PATH}/departments/departments.csv",

    "employees":
        f"{HISTORICAL_BASE_PATH}/employees/employees.csv",

    "suppliers":
        f"{HISTORICAL_BASE_PATH}/suppliers/suppliers.csv",

    "orders":
        f"{HISTORICAL_BASE_PATH}/orders/orders.csv",

    "order_details":
        f"{HISTORICAL_BASE_PATH}/order_details/order_details.csv",

    "payments":
        f"{HISTORICAL_BASE_PATH}/payments/payments.csv",

    "product_suppliers":
        f"{HISTORICAL_BASE_PATH}/product_suppliers/product_suppliers.csv",

    "shippers":
        f"{HISTORICAL_BASE_PATH}/shippers/shippers.csv",

    "shipments":
        f"{HISTORICAL_BASE_PATH}/shipments/shipments.csv"
}


# ================================================================
# 3. READ HISTORICAL TABLES
# ================================================================

source_dfs = {}

print("=" * 80)
print("READING HISTORICAL SOURCE TABLES")
print("=" * 80)

for table_name, path in historical_paths.items():

    print(f"\nReading: {table_name}")
    print(f"Path   : {path}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    source_dfs[table_name] = df

    print(f"Loaded : {table_name}")
    print(f"Rows   : {df.count()}")
    print(f"Cols   : {len(df.columns)}")


# ================================================================
# 4. READ INCREMENTAL API ORDERS
# ================================================================
#
# There are many JSON files:
#
#   order_50001.json
#   order_50002.json
#   ...
#
# We do NOT concatenate them manually.
#
# Spark reads the entire S3 prefix as one logical DataFrame.
#
# ================================================================

print("\n" + "=" * 80)
print("READING INCREMENTAL API ORDERS")
print("=" * 80)

print(f"Path: {API_ORDERS_PATH}")

orders_json_df = (
    spark.read
    .option("multiLine", False)
    .json(API_ORDERS_PATH)
)

source_dfs["orders_json"] = orders_json_df

print("API orders loaded successfully.")
print(f"Rows: {orders_json_df.count()}")
print(f"Columns: {len(orders_json_df.columns)}")


# ================================================================
# 5. CHECK THAT ALL EXPECTED SOURCES WERE LOADED
# ================================================================

expected_sources = [
    "customers",
    "categories",
    "products",
    "departments",
    "employees",
    "suppliers",
    "orders",
    "order_details",
    "payments",
    "product_suppliers",
    "shippers",
    "shipments",
    "orders_json"
]

print("\n" + "=" * 80)
print("SOURCE LOADING CHECK")
print("=" * 80)

for source in expected_sources:

    if source in source_dfs:
        print(f"[OK] {source}")
    else:
        print(f"[MISSING] {source}")


print(f"\nExpected sources: {len(expected_sources)}")
print(f"Loaded sources  : {len(source_dfs)}")


# ================================================================
# 6. DISPLAY SCHEMA FOR EVERY SOURCE
# ================================================================

print("\n" + "=" * 80)
print("SCHEMA FOR ALL SOURCE DATASETS")
print("=" * 80)

for table_name, df in source_dfs.items():

    print("\n" + "-" * 80)
    print(f"SCHEMA: {table_name}")
    print("-" * 80)

    df.printSchema()


# ================================================================
# 7. DISPLAY 10 SAMPLE RECORDS FROM EVERY SOURCE
# ================================================================

print("\n" + "=" * 80)
print("10 SAMPLE RECORDS FROM EACH SOURCE")
print("=" * 80)

for table_name, df in source_dfs.items():

    print("\n" + "-" * 80)
    print(f"SAMPLE RECORDS: {table_name}")
    print("-" * 80)

    df.show(10, truncate=False)


# ================================================================
# 8. BUSINESS / PRIMARY KEYS
# ================================================================
#
# These are the identifiers used to uniquely identify entities
# in the source data.
#
# Product_Suppliers uses a composite business key.
#
# orders_json is nested, so its OrderID will be handled in Part 6.
#
# ================================================================

business_keys = {

    "customers": ["CustomerID"],

    "categories": ["CategoryID"],

    "products": ["ProductID"],

    "departments": ["DepartmentID"],

    "employees": ["EmployeeID"],

    "suppliers": ["SupplierID"],

    "orders": ["OrderID"],

    "order_details": ["OrderDetailID"],

    "payments": ["PaymentID"],

    "product_suppliers": [
        "ProductID",
        "SupplierID"
    ],

    "shippers": ["ShipperID"],

    "shipments": ["ShipmentID"],

    "orders_json": ["OrderID"]
}


print("\n" + "=" * 80)
print("BUSINESS / PRIMARY KEY DEFINITIONS")
print("=" * 80)

for table_name, keys in business_keys.items():

    print(f"{table_name:20} -> {', '.join(keys)}")


# ================================================================
# 9. VALIDATE BUSINESS KEY COLUMNS
# ================================================================
#
# Validation is case-insensitive.
#
# orders_json is nested and therefore intentionally skipped here.
# It will be handled during Part 6 JSON transformation.
#
# ================================================================

print("\n" + "=" * 80)
print("BUSINESS KEY COLUMN VALIDATION")
print("=" * 80)

key_validation = []

for table_name, keys in business_keys.items():

    if table_name == "orders_json":
        key_validation.append(
            (
                table_name,
                "OrderID",
                "NESTED - CHECK IN PART 6"
            )
        )
        continue

    df = source_dfs[table_name]

    actual_columns = {
        column.lower(): column
        for column in df.columns
    }

    for key in keys:

        if key.lower() in actual_columns:

            actual_name = actual_columns[key.lower()]

            key_validation.append(
                (
                    table_name,
                    actual_name,
                    "FOUND"
                )
            )

        else:

            key_validation.append(
                (
                    table_name,
                    key,
                    "MISSING"
                )
            )


key_validation_df = spark.createDataFrame(
    key_validation,
    [
        "table_name",
        "key_column",
        "status"
    ]
)

key_validation_df.show(
    truncate=False
)


# ================================================================
# 10. FOREIGN KEY RELATIONSHIPS
# ================================================================
#
# Documented relationships between source tables.
#
# ================================================================

foreign_keys = [

    (
        "orders",
        "CustomerID",
        "customers",
        "CustomerID"
    ),

    (
        "products",
        "CategoryID",
        "categories",
        "CategoryID"
    ),

    (
        "products",
        "DepartmentID",
        "departments",
        "DepartmentID"
    ),

    (
        "employees",
        "DepartmentID",
        "departments",
        "DepartmentID"
    ),

    (
        "order_details",
        "OrderID",
        "orders",
        "OrderID"
    ),

    (
        "order_details",
        "ProductID",
        "products",
        "ProductID"
    ),

    (
        "payments",
        "OrderID",
        "orders",
        "OrderID"
    ),

    (
        "product_suppliers",
        "ProductID",
        "products",
        "ProductID"
    ),

    (
        "product_suppliers",
        "SupplierID",
        "suppliers",
        "SupplierID"
    ),

    (
        "shipments",
        "OrderID",
        "orders",
        "OrderID"
    ),

    (
        "shipments",
        "ShipperID",
        "shippers",
        "ShipperID"
    )
]


fk_df = spark.createDataFrame(
    foreign_keys,
    [
        "child_table",
        "child_column",
        "parent_table",
        "parent_column"
    ]
)


print("\n" + "=" * 80)
print("FOREIGN KEY RELATIONSHIPS")
print("=" * 80)

fk_df.show(
    truncate=False
)


# ================================================================
# 11. RECORD AND COLUMN COUNTS
# ================================================================

print("\n" + "=" * 80)
print("RECORD AND COLUMN COUNTS")
print("=" * 80)

profile_rows = []

for table_name, df in source_dfs.items():

    row_count = df.count()
    column_count = len(df.columns)

    profile_rows.append(
        (
            table_name,
            row_count,
            column_count
        )
    )


profile_df = spark.createDataFrame(
    profile_rows,
    [
        "table_name",
        "record_count",
        "column_count"
    ]
)


profile_df.orderBy(
    "table_name"
).show(
    truncate=False
)


# ================================================================
# 12. NULLABLE COLUMN METADATA
# ================================================================
#
# StructField.nullable tells us whether Spark's schema allows
# NULL values for the column.
#
# This is schema-level nullability.
#
# Actual NULL counts will be investigated in Part 2.
#
# ================================================================

print("\n" + "=" * 80)
print("NULLABLE COLUMN METADATA")
print("=" * 80)

nullable_rows = []

for table_name, df in source_dfs.items():

    for field in df.schema.fields:

        nullable_rows.append(
            (
                table_name,
                field.name,
                str(field.dataType),
                field.nullable
            )
        )


nullable_df = spark.createDataFrame(
    nullable_rows,
    [
        "table_name",
        "column_name",
        "data_type",
        "nullable"
    ]
)


nullable_df.show(
    200,
    truncate=False
)


# ================================================================
# 13. FINAL PART 1 SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("PART 1 SUMMARY")
print("=" * 80)

print(f"Number of source datasets : {len(source_dfs)}")
print(f"Historical datasets       : {len(historical_paths)}")
print("Incremental JSON dataset  : orders_json")

print("\nBusiness keys:")
for table_name, keys in business_keys.items():
    print(
        f"  {table_name:20} -> {', '.join(keys)}"
    )

print("\nForeign-key relationships:")
print(f"  {len(foreign_keys)} relationships documented.")

print("\nObjects are being processed directly from S3.")
print("No manual concatenation of API JSON files is required.")

print("\n" + "=" * 80)
print("PART 1 — DATA UNDERSTANDING COMPLETED")
print("=" * 80)

READING HISTORICAL SOURCE TABLES

Reading: customers
Path   : s3://aws-ecommerce-s3/ecommerce/raw/customers/customers.csv
Loaded : customers
Rows   : 10000
Cols   : 7

Reading: categories
Path   : s3://aws-ecommerce-s3/ecommerce/raw/categories/categories.csv
Loaded : categories
Rows   : 20
Cols   : 2

Reading: products
Path   : s3://aws-ecommerce-s3/ecommerce/raw/products/products.csv
Loaded : products
Rows   : 1000
Cols   : 7

Reading: departments
Path   : s3://aws-ecommerce-s3/ecommerce/raw/departments/departments.csv
Loaded : departments
Rows   : 10
Cols   : 2

Reading: employees
Path   : s3://aws-ecommerce-s3/ecommerce/raw/employees/employees.csv
Loaded : employees
Rows   : 200
Cols   : 7

Reading: suppliers
Path   : s3://aws-ecommerce-s3/ecommerce/raw/suppliers/suppliers.csv
Loaded : suppliers
Rows   : 100
Cols   : 3

Reading: orders
Path   : s3://aws-ecommerce-s3/ecommerce/raw/orders/orders.csv
Loaded : orders
Rows   : 50000
Cols   : 4

Reading: order_details
Path   : s3://aws-ec

In [6]:
# ================================================================
# PART 2 — DATA QUALITY
# ================================================================
#
# Purpose:
#   Task 2 — Identify NULL values
#   Task 3 — Identify duplicate records using business keys
#   Task 4 — Identify invalid values and broken relationships
#
# IMPORTANT:
#   This cell uses the DataFrames created in Part 1:
#
#       source_dfs
#
#   No data is reread from S3.
#
#   This part ONLY identifies data-quality problems.
#   Cleaning/removing/fixing the problems will be performed in Part 3.
#
# ================================================================


from pyspark.sql import functions as F


# ================================================================
# 0. BASIC CHECK
# ================================================================

print("=" * 80)
print("PART 2 — DATA QUALITY")
print("=" * 80)

print(f"Datasets available from Part 1: {len(source_dfs)}")

for name in source_dfs:
    print(f"  [OK] {name}")


# ================================================================
# HELPER FUNCTION — FIND COLUMN CASE-INSENSITIVELY
# ================================================================
#
# Example:
#   If the actual column is "customerid", this function can still
#   find it when we request "CustomerID".
#
# ================================================================

def find_column(df, expected_name):

    for actual_name in df.columns:

        if actual_name.lower() == expected_name.lower():
            return actual_name

    return None


# ================================================================
# HELPER FUNCTION — CHECK THAT A DATASET HAS A COLUMN
# ================================================================

def get_column_or_none(table_name, expected_name):

    if table_name not in source_dfs:
        return None

    return find_column(
        source_dfs[table_name],
        expected_name
    )


# ================================================================
# ================================================================
# TASK 2 — NULL VALUES
# ================================================================
#
# Requirement:
#
#   Identify columns containing NULL values.
#
#   Report:
#       table
#       column
#       null_count
#       null_percentage
#
# ================================================================

print("\n" + "=" * 80)
print("TASK 2 — NULL VALUES")
print("=" * 80)


null_results = []


for table_name, df in source_dfs.items():

    print(f"\nChecking NULL values: {table_name}")

    # ------------------------------------------------------------
    # Count records once for this table
    # ------------------------------------------------------------

    total_rows = df.count()

    if total_rows == 0:

        print("  Table contains 0 records.")
        continue

    # ------------------------------------------------------------
    # Calculate NULL count for every column
    # ------------------------------------------------------------

    null_count_expressions = []

    for column_name in df.columns:

        null_count_expressions.append(
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(column_name)
        )

    null_counts_row = (
        df.select(*null_count_expressions)
        .collect()[0]
    )

    # ------------------------------------------------------------
    # Convert results into rows
    # ------------------------------------------------------------

    for column_name in df.columns:

        null_count = null_counts_row[column_name]

        if null_count is None:
            null_count = 0

        if null_count > 0:

            null_percentage = (
                null_count / total_rows
            ) * 100

            null_results.append(
                (
                    table_name,
                    column_name,
                    int(null_count),
                    float(null_percentage)
                )
            )


# ---------------------------------------------------------------
# Create NULL quality report
# ---------------------------------------------------------------

if null_results:

    null_quality_df = spark.createDataFrame(
        null_results,
        [
            "table",
            "column",
            "null_count",
            "null_percentage"
        ]
    )

    null_quality_df = (
        null_quality_df
        .orderBy(
            F.col("table"),
            F.desc("null_count")
        )
    )

    print("\nColumns containing NULL values:")
    null_quality_df.show(
        500,
        truncate=False
    )

else:

    null_quality_df = spark.createDataFrame(
        [],
        """
        table string,
        column string,
        null_count long,
        null_percentage double
        """
    )

    print("\nNo NULL values were found.")


# ---------------------------------------------------------------
# NULL summary
# ---------------------------------------------------------------

print("\nNULL columns found:", len(null_results))


# ================================================================
# ================================================================
# TASK 3 — DUPLICATE RECORDS
# ================================================================
#
# Requirement:
#
# Find duplicate records based on the appropriate business key
# for:
#
#   customers
#   products
#   categories
#   departments
#   employees
#   suppliers
#   orders
#   order_details
#   payments
#   shippers
#   shipments
#
# ================================================================


print("\n" + "=" * 80)
print("TASK 3 — DUPLICATE RECORDS")
print("=" * 80)


# ---------------------------------------------------------------
# Business keys from Part 1
# ---------------------------------------------------------------

duplicate_keys = {

    "customers": ["CustomerID"],

    "products": ["ProductID"],

    "categories": ["CategoryID"],

    "departments": ["DepartmentID"],

    "employees": ["EmployeeID"],

    "suppliers": ["SupplierID"],

    "orders": ["OrderID"],

    "order_details": ["OrderDetailID"],

    "payments": ["PaymentID"],

    "shippers": ["ShipperID"],

    "shipments": ["ShipmentID"]
}


duplicate_results = []


for table_name, key_columns in duplicate_keys.items():

    df = source_dfs[table_name]

    # ------------------------------------------------------------
    # Resolve actual column names
    # ------------------------------------------------------------

    actual_key_columns = []

    missing_keys = []

    for key in key_columns:

        actual_key = find_column(
            df,
            key
        )

        if actual_key is None:
            missing_keys.append(key)
        else:
            actual_key_columns.append(actual_key)

    # ------------------------------------------------------------
    # Handle missing business key
    # ------------------------------------------------------------

    if missing_keys:

        print(
            f"[WARNING] {table_name}: "
            f"missing key column(s): {missing_keys}"
        )

        continue

    # ------------------------------------------------------------
    # Ignore rows where the business key is NULL.
    #
    # NULL-key problems are already reported by Task 2.
    # ------------------------------------------------------------

    non_null_condition = None

    for key_column in actual_key_columns:

        condition = F.col(key_column).isNotNull()

        if non_null_condition is None:
            non_null_condition = condition
        else:
            non_null_condition = (
                non_null_condition & condition
            )

    keyed_df = df.filter(non_null_condition)

    # ------------------------------------------------------------
    # Find business keys occurring more than once
    # ------------------------------------------------------------

    duplicate_groups = (
        keyed_df
        .groupBy(*actual_key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_group_count = duplicate_groups.count()

    # ------------------------------------------------------------
    # Number of rows involved in duplicate groups
    #
    # Example:
    # ID 10 occurs 3 times
    #
    # duplicate rows beyond first = 2
    # ------------------------------------------------------------

    if duplicate_group_count > 0:

        duplicate_rows = (
            duplicate_groups
            .select(
                F.sum(
                    F.col("count") - 1
                ).alias("duplicate_rows")
            )
            .collect()[0]["duplicate_rows"]
        )

        if duplicate_rows is None:
            duplicate_rows = 0

    else:

        duplicate_rows = 0

    # ------------------------------------------------------------
    # Store result
    # ------------------------------------------------------------

    duplicate_results.append(
        (
            table_name,
            ", ".join(actual_key_columns),
            int(duplicate_group_count),
            int(duplicate_rows)
        )
    )


# ---------------------------------------------------------------
# Create duplicate report
# ---------------------------------------------------------------

duplicate_quality_df = spark.createDataFrame(
    duplicate_results,
    [
        "table",
        "business_key",
        "duplicate_key_groups",
        "duplicate_rows_beyond_first"
    ]
)


duplicate_quality_df = (
    duplicate_quality_df
    .orderBy("table")
)


print("\nDuplicate summary:")
duplicate_quality_df.show(
    truncate=False
)


# ================================================================
# DUPLICATE HANDLING STRATEGY
# ================================================================

print("\n" + "-" * 80)
print("DUPLICATE HANDLING STRATEGY")
print("-" * 80)

print("""
1. Duplicates are identified using the appropriate business key.

2. In Part 3, duplicate records will be removed so that one
   record remains for each business key.

3. Rows with NULL business keys are handled separately as NULL
   data-quality problems.

4. If duplicate rows contain conflicting attribute values,
   they should not be arbitrarily merged. The selected record
   should follow a documented business rule when such a rule
   exists.

5. The duplicate detection in this part does NOT modify the
   source DataFrames.
""")


# ================================================================
# ================================================================
# TASK 4 — INVALID VALUES
# ================================================================
#
# Identify:
#
#   1. Negative product prices
#   2. Negative quantities
#   3. Negative payment amounts
#   4. Invalid email addresses
#   5. Invalid dates
#   6. Delivery before shipment
#   7. Orders referencing nonexistent customers
#   8. Order details referencing nonexistent products
#   9. Payments referencing nonexistent orders
#  10. Shipments referencing nonexistent orders
#
# ================================================================


print("\n" + "=" * 80)
print("TASK 4 — INVALID VALUES")
print("=" * 80)


# ================================================================
# 4.1 NEGATIVE PRODUCT PRICES
# ================================================================

print("\n" + "-" * 80)
print("4.1 — NEGATIVE PRODUCT PRICES")
print("-" * 80)


products_df = source_dfs["products"]

product_price_column = find_column(
    products_df,
    "Price"
)


if product_price_column:

    negative_product_prices_df = (
        products_df
        .filter(
            F.col(product_price_column) < 0
        )
    )

    negative_product_price_count = (
        negative_product_prices_df.count()
    )

    print(
        f"Negative product-price records: "
        f"{negative_product_price_count}"
    )

    if negative_product_price_count > 0:

        negative_product_prices_df.show(
            20,
            truncate=False
        )

else:

    negative_product_price_count = 0

    print(
        "[WARNING] Price column was not found "
        "in products."
    )


# ================================================================
# 4.2 NEGATIVE QUANTITIES
# ================================================================

print("\n" + "-" * 80)
print("4.2 — NEGATIVE QUANTITIES")
print("-" * 80)


order_details_df = source_dfs["order_details"]

quantity_column = find_column(
    order_details_df,
    "Quantity"
)


if quantity_column:

    negative_quantities_df = (
        order_details_df
        .filter(
            F.col(quantity_column) < 0
        )
    )

    negative_quantity_count = (
        negative_quantities_df.count()
    )

    print(
        f"Negative-quantity records: "
        f"{negative_quantity_count}"
    )

    if negative_quantity_count > 0:

        negative_quantities_df.show(
            20,
            truncate=False
        )

else:

    negative_quantity_count = 0

    print(
        "[WARNING] Quantity column was not found "
        "in order_details."
    )


# ================================================================
# 4.3 NEGATIVE PAYMENT AMOUNTS
# ================================================================

print("\n" + "-" * 80)
print("4.3 — NEGATIVE PAYMENT AMOUNTS")
print("-" * 80)


payments_df = source_dfs["payments"]

payment_amount_column = find_column(
    payments_df,
    "Amount"
)


if payment_amount_column:

    negative_payment_amounts_df = (
        payments_df
        .filter(
            F.col(payment_amount_column) < 0
        )
    )

    negative_payment_amount_count = (
        negative_payment_amounts_df.count()
    )

    print(
        f"Negative-payment records: "
        f"{negative_payment_amount_count}"
    )

    if negative_payment_amount_count > 0:

        negative_payment_amounts_df.show(
            20,
            truncate=False
        )

else:

    negative_payment_amount_count = 0

    print(
        "[WARNING] Amount column was not found "
        "in payments."
    )


# ================================================================
# 4.4 INVALID EMAIL ADDRESSES
# ================================================================
#
# Basic structural validation:
#
#   something@domain.extension
#
# This is NOT an email-existence check.
# It only identifies values that do not match a basic email format.
#
# ================================================================

print("\n" + "-" * 80)
print("4.4 — INVALID EMAIL ADDRESSES")
print("-" * 80)


customers_df = source_dfs["customers"]

email_column = find_column(
    customers_df,
    "Email"
)


if email_column:

    email_pattern = (
        r"^[A-Za-z0-9._%+-]+"
        r"@[A-Za-z0-9.-]+\."
        r"[A-Za-z]{2,}$"
    )

    invalid_email_df = (
        customers_df
        .filter(
            F.col(email_column).isNotNull()
            &
            (F.trim(F.col(email_column)) != "")
            &
            (~F.col(email_column).rlike(email_pattern))
        )
    )

    invalid_email_count = (
        invalid_email_df.count()
    )

    print(
        f"Invalid-email records: "
        f"{invalid_email_count}"
    )

    if invalid_email_count > 0:

        invalid_email_df.show(
            20,
            truncate=False
        )

else:

    invalid_email_count = 0

    print(
        "[WARNING] Email column was not found "
        "in customers."
    )


# ================================================================
# 4.5 INVALID DATES
# ================================================================
#
# We check the date fields used by the source model.
#
# A value is considered an invalid date when:
#
#   - it is not NULL
#   - it is not empty
#   - Spark cannot parse it as a date
#
# Expected date fields:
#
#   customers       -> RegistrationDate
#   orders          -> OrderDate
#   payments        -> PaymentDate
#   shipments       -> ShipDate
#   shipments       -> DeliveryDate
#
# ================================================================

print("\n" + "-" * 80)
print("4.5 — INVALID DATES")
print("-" * 80)


date_columns_by_table = {

    "customers": ["RegistrationDate"],

    "orders": ["OrderDate"],

    "payments": ["PaymentDate"],

    "shipments": [
        "ShipDate",
        "DeliveryDate"
    ]
}


invalid_date_results = []


for table_name, expected_date_columns in date_columns_by_table.items():

    df = source_dfs[table_name]

    for expected_date_column in expected_date_columns:

        actual_date_column = find_column(
            df,
            expected_date_column
        )

        if actual_date_column is None:

            print(
                f"[WARNING] {table_name}.{expected_date_column} "
                f"was not found."
            )

            continue

        # --------------------------------------------------------
        # Convert to string first so both date and string columns
        # can be handled.
        # --------------------------------------------------------

        raw_value = F.trim(
            F.col(actual_date_column).cast("string")
        )

        parsed_date = F.to_date(
            raw_value
        )

        invalid_date_condition = (
            F.col(actual_date_column).isNotNull()
            &
            (raw_value != "")
            &
            parsed_date.isNull()
        )

        invalid_df = df.filter(
            invalid_date_condition
        )

        invalid_count = invalid_df.count()

        invalid_date_results.append(
            (
                table_name,
                actual_date_column,
                int(invalid_count)
            )
        )

        print(
            f"{table_name}.{actual_date_column}: "
            f"{invalid_count} invalid dates"
        )

        if invalid_count > 0:

            invalid_df.show(
                20,
                truncate=False
            )


invalid_dates_df = spark.createDataFrame(
    invalid_date_results,
    [
        "table",
        "column",
        "invalid_date_count"
    ]
)


print("\nInvalid-date summary:")

invalid_dates_df.show(
    truncate=False
)


# ================================================================
# 4.6 SHIPMENTS DELIVERED BEFORE THEY WERE SHIPPED
# ================================================================

print("\n" + "-" * 80)
print("4.6 — DELIVERY BEFORE SHIPMENT")
print("-" * 80)


shipments_df = source_dfs["shipments"]

ship_date_column = find_column(
    shipments_df,
    "ShipDate"
)

delivery_date_column = find_column(
    shipments_df,
    "DeliveryDate"
)


if ship_date_column and delivery_date_column:

    shipments_with_dates_df = (
        shipments_df
        .withColumn(
            "_ship_date",
            F.to_date(
                F.col(ship_date_column).cast("string")
            )
        )
        .withColumn(
            "_delivery_date",
            F.to_date(
                F.col(delivery_date_column).cast("string")
            )
        )
    )

    invalid_shipment_dates_df = (
        shipments_with_dates_df
        .filter(
            F.col("_ship_date").isNotNull()
            &
            F.col("_delivery_date").isNotNull()
            &
            (
                F.col("_delivery_date")
                <
                F.col("_ship_date")
            )
        )
        .drop(
            "_ship_date",
            "_delivery_date"
        )
    )

    invalid_shipment_date_count = (
        invalid_shipment_dates_df.count()
    )

    print(
        "Shipments with delivery before shipment: "
        f"{invalid_shipment_date_count}"
    )

    if invalid_shipment_date_count > 0:

        invalid_shipment_dates_df.show(
            20,
            truncate=False
        )

else:

    invalid_shipment_date_count = 0

    print(
        "[WARNING] ShipDate and/or DeliveryDate "
        "was not found in shipments."
    )


# ================================================================
# 4.7 ORDERS REFERENCING NONEXISTENT CUSTOMERS
# ================================================================

print("\n" + "-" * 80)
print("4.7 — ORPHAN ORDERS / NONEXISTENT CUSTOMERS")
print("-" * 80)


orders_df = source_dfs["orders"]

order_customer_column = find_column(
    orders_df,
    "CustomerID"
)

customer_key_column = find_column(
    customers_df,
    "CustomerID"
)


if order_customer_column and customer_key_column:

    customer_keys_df = (
        customers_df
        .select(
            F.col(customer_key_column)
            .alias("_CustomerID")
        )
        .filter(
            F.col("_CustomerID").isNotNull()
        )
        .dropDuplicates()
    )

    orphan_orders_df = (
        orders_df.alias("o")
        .join(
            customer_keys_df.alias("c"),
            F.col(f"o.{order_customer_column}")
            ==
            F.col("c._CustomerID"),
            "left_anti"
        )
    )

    orphan_order_count = (
        orphan_orders_df.count()
    )

    print(
        "Orders referencing nonexistent customers: "
        f"{orphan_order_count}"
    )

    if orphan_order_count > 0:

        orphan_orders_df.show(
            20,
            truncate=False
        )

else:

    orphan_order_count = 0

    print(
        "[WARNING] CustomerID could not be resolved "
        "in orders/customers."
    )


# ================================================================
# 4.8 ORDER DETAILS REFERENCING NONEXISTENT PRODUCTS
# ================================================================

print("\n" + "-" * 80)
print("4.8 — ORPHAN ORDER DETAILS / NONEXISTENT PRODUCTS")
print("-" * 80)


order_detail_product_column = find_column(
    order_details_df,
    "ProductID"
)

product_key_column = find_column(
    products_df,
    "ProductID"
)


if order_detail_product_column and product_key_column:

    product_keys_df = (
        products_df
        .select(
            F.col(product_key_column)
            .alias("_ProductID")
        )
        .filter(
            F.col("_ProductID").isNotNull()
        )
        .dropDuplicates()
    )

    orphan_order_details_df = (
        order_details_df.alias("od")
        .join(
            product_keys_df.alias("p"),
            F.col(
                f"od.{order_detail_product_column}"
            )
            ==
            F.col("p._ProductID"),
            "left_anti"
        )
    )

    orphan_order_detail_count = (
        orphan_order_details_df.count()
    )

    print(
        "Order details referencing nonexistent products: "
        f"{orphan_order_detail_count}"
    )

    if orphan_order_detail_count > 0:

        orphan_order_details_df.show(
            20,
            truncate=False
        )

else:

    orphan_order_detail_count = 0

    print(
        "[WARNING] ProductID could not be resolved "
        "in order_details/products."
    )


# ================================================================
# 4.9 PAYMENTS REFERENCING NONEXISTENT ORDERS
# ================================================================

print("\n" + "-" * 80)
print("4.9 — ORPHAN PAYMENTS / NONEXISTENT ORDERS")
print("-" * 80)


payment_order_column = find_column(
    payments_df,
    "OrderID"
)

order_key_column = find_column(
    orders_df,
    "OrderID"
)


if payment_order_column and order_key_column:

    order_keys_df = (
        orders_df
        .select(
            F.col(order_key_column)
            .alias("_OrderID")
        )
        .filter(
            F.col("_OrderID").isNotNull()
        )
        .dropDuplicates()
    )

    orphan_payments_df = (
        payments_df.alias("pay")
        .join(
            order_keys_df.alias("ord"),
            F.col(
                f"pay.{payment_order_column}"
            )
            ==
            F.col("ord._OrderID"),
            "left_anti"
        )
    )

    orphan_payment_count = (
        orphan_payments_df.count()
    )

    print(
        "Payments referencing nonexistent orders: "
        f"{orphan_payment_count}"
    )

    if orphan_payment_count > 0:

        orphan_payments_df.show(
            20,
            truncate=False
        )

else:

    orphan_payment_count = 0

    print(
        "[WARNING] OrderID could not be resolved "
        "in payments/orders."
    )


# ================================================================
# 4.10 SHIPMENTS REFERENCING NONEXISTENT ORDERS
# ================================================================

print("\n" + "-" * 80)
print("4.10 — ORPHAN SHIPMENTS / NONEXISTENT ORDERS")
print("-" * 80)


shipment_order_column = find_column(
    shipments_df,
    "OrderID"
)


if shipment_order_column and order_key_column:

    order_keys_df = (
        orders_df
        .select(
            F.col(order_key_column)
            .alias("_OrderID")
        )
        .filter(
            F.col("_OrderID").isNotNull()
        )
        .dropDuplicates()
    )

    orphan_shipments_df = (
        shipments_df.alias("ship")
        .join(
            order_keys_df.alias("ord"),
            F.col(
                f"ship.{shipment_order_column}"
            )
            ==
            F.col("ord._OrderID"),
            "left_anti"
        )
    )

    orphan_shipment_count = (
        orphan_shipments_df.count()
    )

    print(
        "Shipments referencing nonexistent orders: "
        f"{orphan_shipment_count}"
    )

    if orphan_shipment_count > 0:

        orphan_shipments_df.show(
            20,
            truncate=False
        )

else:

    orphan_shipment_count = 0

    print(
        "[WARNING] OrderID could not be resolved "
        "in shipments/orders."
    )


# ================================================================
# ================================================================
# TASK 4 — FINAL INVALID VALUE SUMMARY
# ================================================================
# ================================================================

invalid_value_summary = [

    (
        "Negative product prices",
        int(negative_product_price_count)
    ),

    (
        "Negative quantities",
        int(negative_quantity_count)
    ),

    (
        "Negative payment amounts",
        int(negative_payment_amount_count)
    ),

    (
        "Invalid email addresses",
        int(invalid_email_count)
    ),

    (
        "Shipments delivered before shipment",
        int(invalid_shipment_date_count)
    ),

    (
        "Orders with nonexistent customers",
        int(orphan_order_count)
    ),

    (
        "Order details with nonexistent products",
        int(orphan_order_detail_count)
    ),

    (
        "Payments with nonexistent orders",
        int(orphan_payment_count)
    ),

    (
        "Shipments with nonexistent orders",
        int(orphan_shipment_count)
    )
]


invalid_value_summary_df = spark.createDataFrame(
    invalid_value_summary,
    [
        "check",
        "invalid_record_count"
    ]
)


print("\n" + "=" * 80)
print("INVALID VALUE SUMMARY")
print("=" * 80)

invalid_value_summary_df.show(
    truncate=False
)


# ================================================================
# FINAL PART 2 SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("PART 2 — DATA QUALITY COMPLETED")
print("=" * 80)

print("""
Task 2:
    NULL values identified and reported by table and column.

Task 3:
    Duplicate business keys identified for the required tables.

Task 4:
    Invalid numeric values, email formats, dates, date ordering,
    and referential-integrity violations were checked.

No source data was modified in Part 2.

Cleaning and duplicate removal will be performed in Part 3.
""")

print("=" * 80)

PART 2 — DATA QUALITY
Datasets available from Part 1: 13
  [OK] customers
  [OK] categories
  [OK] products
  [OK] departments
  [OK] employees
  [OK] suppliers
  [OK] orders
  [OK] order_details
  [OK] payments
  [OK] product_suppliers
  [OK] shippers
  [OK] shipments
  [OK] orders_json

TASK 2 — NULL VALUES

Checking NULL values: customers

Checking NULL values: categories

Checking NULL values: products

Checking NULL values: departments

Checking NULL values: employees

Checking NULL values: suppliers

Checking NULL values: orders

Checking NULL values: order_details

Checking NULL values: payments

Checking NULL values: product_suppliers

Checking NULL values: shippers

Checking NULL values: shipments

Checking NULL values: orders_json

Columns containing NULL values:
+-----------+---------+----------+-----------------+
|table      |column   |null_count|null_percentage  |
+-----------+---------+----------+-----------------+
|employees  |ManagerID|1         |0.5              |
|ord

In [8]:
# ================================================================
# PART 3 — DATA CLEANING
# ================================================================
#
# Task 5:
#
#   1. Standardize column names
#   2. Remove unnecessary spaces
#   3. Standardize string values
#   4. Convert columns to appropriate data types
#   5. Clean email addresses
#   6. Handle NULL values
#   7. Remove duplicate records
#   8. Validate dates
#   9. Validate numeric columns
#  10. Save cleaned data as Parquet
#
# IMPORTANT:
#   - Uses source_dfs created in Part 1.
#   - Does NOT reread the raw S3 data.
#   - Does NOT modify the raw source.
#   - orders_json is intentionally excluded from the relational
#     cleaning here. Its nested transformation is handled in Part 6.
#
# Output:
#   s3://aws-ecommerce-s3/ecommerce/processed/
#
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    DateType
)


# ================================================================
# 0. CONFIGURATION
# ================================================================

# Actual bucket used by your current AWS environment.
#
# Assignment PDF path:
# s3://aws-ecommerce-data-omar-2026/processed/ecommerce/
#
# Your actual bucket:
# s3://aws-ecommerce-s3/ecommerce/processed/

PROCESSED_PATH = (
    "s3://aws-ecommerce-s3/ecommerce/processed"
)


# Historical relational tables.
#
# orders_json will be processed separately in Part 6.

cleaning_tables = [
    "customers",
    "categories",
    "products",
    "departments",
    "employees",
    "suppliers",
    "orders",
    "order_details",
    "payments",
    "product_suppliers",
    "shippers",
    "shipments"
]


print("=" * 80)
print("PART 3 — DATA CLEANING")
print("=" * 80)

print(f"Input datasets available: {len(source_dfs)}")
print(f"Output path: {PROCESSED_PATH}")


# ================================================================
# 1. HELPER — STANDARDIZE COLUMN NAME
# ================================================================
#
# Examples:
#
#   CustomerID       -> customer_id
#   FirstName        -> first_name
#   OrderDate        -> order_date
#   CompanyName      -> company_name
#
# ================================================================

def standardize_column_name(column_name):

    name = column_name.strip()

    # Insert underscore between:
    # lower-case/digit followed by upper-case
    import re

    name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        name
    )

    # Replace spaces and special characters
    name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        name
    )

    # Remove duplicate underscores
    name = re.sub(
        r"_+",
        "_",
        name
    )

    # Remove leading/trailing underscores
    name = name.strip("_")

    return name.lower()


# ================================================================
# 2. HELPER — STANDARDIZE ALL COLUMN NAMES
# ================================================================

def standardize_column_names(df):

    for old_name in df.columns:

        new_name = standardize_column_name(
            old_name
        )

        if old_name != new_name:

            df = df.withColumnRenamed(
                old_name,
                new_name
            )

    return df


# ================================================================
# 3. HELPER — FIND COLUMN AFTER STANDARDIZATION
# ================================================================

def has_column(df, column_name):

    return column_name.lower() in [
        c.lower()
        for c in df.columns
    ]


# ================================================================
# 4. HELPER — TRIM ALL STRING COLUMNS
# ================================================================

def trim_string_columns(df):

    for field in df.schema.fields:

        if isinstance(field.dataType, StringType):

            df = df.withColumn(
                field.name,
                F.trim(
                    F.col(field.name)
                )
            )

    return df


# ================================================================
# 5. HELPER — STANDARDIZE COMMON STRING VALUES
# ================================================================
#
# We do not blindly uppercase/lowercase every string because
# names, brands, cities, categories, etc. are case-sensitive
# presentation values.
#
# We:
#   - trim whitespace
#   - replace empty strings with NULL
#   - standardize status fields to uppercase
#   - standardize email to lowercase
#
# ================================================================

def standardize_string_values(df):

    # ------------------------------------------------------------
    # Convert empty/whitespace-only strings to NULL
    # ------------------------------------------------------------

    for field in df.schema.fields:

        if isinstance(field.dataType, StringType):

            df = df.withColumn(
                field.name,
                F.when(
                    F.trim(F.col(field.name)) == "",
                    None
                ).otherwise(
                    F.trim(F.col(field.name))
                )
            )

    # ------------------------------------------------------------
    # Standardize email fields
    # ------------------------------------------------------------

    for column_name in [
        "email"
    ]:

        if has_column(df, column_name):

            df = df.withColumn(
                column_name,
                F.lower(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )

    # ------------------------------------------------------------
    # Standardize status fields
    # ------------------------------------------------------------

    for column_name in [
        "status",
        "payment_status"
    ]:

        if has_column(df, column_name):

            df = df.withColumn(
                column_name,
                F.upper(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )

    return df


# ================================================================
# 6. EXPECTED DATA TYPES
# ================================================================
#
# These mappings correspond to the source model.
#
# Only columns that actually exist in a table are converted.
#
# ================================================================

expected_types = {

    "customers": {
        "customer_id": IntegerType(),
        "first_name": StringType(),
        "last_name": StringType(),
        "email": StringType(),
        "city": StringType(),
        "country": StringType(),
        "registration_date": DateType()
    },

    "categories": {
        "category_id": IntegerType(),
        "category_name": StringType()
    },

    "products": {
        "product_id": IntegerType(),
        "category_id": IntegerType(),
        "department_id": IntegerType(),
        "product_name": StringType(),
        "price": DoubleType()
    },

    "departments": {
        "department_id": IntegerType(),
        "department_name": StringType()
    },

    "employees": {
        "employee_id": IntegerType(),
        "first_name": StringType(),
        "last_name": StringType(),
        "email": StringType(),
        "department_id": IntegerType()
    },

    "suppliers": {
        "supplier_id": IntegerType(),
        "company_name": StringType(),
        "contact_name": StringType(),
        "email": StringType()
    },

    "orders": {
        "order_id": IntegerType(),
        "customer_id": IntegerType(),
        "order_date": DateType(),
        "status": StringType()
    },

    "order_details": {
        "order_detail_id": IntegerType(),
        "order_id": IntegerType(),
        "product_id": IntegerType(),
        "quantity": IntegerType(),
        "unit_price": DoubleType(),
        "discount": DoubleType()
    },

    "payments": {
        "payment_id": IntegerType(),
        "order_id": IntegerType(),
        "payment_method": StringType(),
        "payment_date": DateType(),
        "amount": DoubleType(),
        "payment_status": StringType()
    },

    "product_suppliers": {
        "product_id": IntegerType(),
        "supplier_id": IntegerType()
    },

    "shippers": {
        "shipper_id": IntegerType(),
        "company_name": StringType()
    },

    "shipments": {
        "shipment_id": IntegerType(),
        "order_id": IntegerType(),
        "shipper_id": IntegerType(),
        "ship_date": DateType(),
        "delivery_date": DateType()
    }
}


# ================================================================
# 7. BUSINESS KEYS
# ================================================================
#
# Used for duplicate removal and NULL-key handling.
#
# ================================================================

cleaning_keys = {

    "customers": ["customer_id"],

    "categories": ["category_id"],

    "products": ["product_id"],

    "departments": ["department_id"],

    "employees": ["employee_id"],

    "suppliers": ["supplier_id"],

    "orders": ["order_id"],

    "order_details": ["order_detail_id"],

    "payments": ["payment_id"],

    "product_suppliers": [
        "product_id",
        "supplier_id"
    ],

    "shippers": ["shipper_id"],

    "shipments": ["shipment_id"]
}


# ================================================================
# 8. EMAIL VALIDATION
# ================================================================
#
# Invalid email formats are converted to NULL.
#
# Example:
#
#   " USER@Example.COM " -> "user@example.com"
#
# Invalid:
#
#   "userexample.com" -> NULL
#
# ================================================================

email_pattern = (
    r"^[A-Za-z0-9._%+-]+"
    r"@[A-Za-z0-9.-]+\."
    r"[A-Za-z]{2,}$"
)


# ================================================================
# 9. PROCESS EACH TABLE
# ================================================================

cleaned_dfs = {}

cleaning_summary = []


for table_name in cleaning_tables:

    print("\n" + "=" * 80)
    print(f"CLEANING TABLE: {table_name}")
    print("=" * 80)

    df = source_dfs[table_name]

    original_count = df.count()
    original_columns = len(df.columns)

    print(f"Original rows    : {original_count}")
    print(f"Original columns : {original_columns}")


    # ============================================================
    # STEP 1 — STANDARDIZE COLUMN NAMES
    # ============================================================

    df = standardize_column_names(df)

    print("\n[1] Column names standardized")


    # ============================================================
    # STEP 2 — TRIM SPACES
    # ============================================================

    df = trim_string_columns(df)

    print("[2] Unnecessary spaces removed")


    # ============================================================
    # STEP 3 — STANDARDIZE STRING VALUES
    # ============================================================

    df = standardize_string_values(df)

    print("[3] String values standardized")


    # ============================================================
    # STEP 4 — CONVERT DATA TYPES
    # ============================================================

    type_mapping = expected_types.get(
        table_name,
        {}
    )

    for column_name, target_type in type_mapping.items():

        if has_column(df, column_name):

            if isinstance(
                target_type,
                DateType
            ):

                # Explicit date conversion
                df = df.withColumn(
                    column_name,
                    F.to_date(
                        F.col(column_name).cast("string")
                    )
                )

            else:

                df = df.withColumn(
                    column_name,
                    F.col(column_name).cast(
                        target_type
                    )
                )

    print("[4] Data types converted")


    # ============================================================
    # STEP 5 — CLEAN EMAIL ADDRESSES
    # ============================================================

    if has_column(df, "email"):

        df = df.withColumn(
            "email",
            F.lower(
                F.trim(
                    F.col("email")
                )
            )
        )

        df = df.withColumn(
            "email",
            F.when(
                F.col("email").isNull(),
                None
            )
            .when(
                F.col("email") == "",
                None
            )
            .when(
                F.col("email").rlike(
                    email_pattern
                ),
                F.col("email")
            )
            .otherwise(
                None
            )
        )

        print(
            "[5] Email addresses trimmed, "
            "lowercased, and validated"
        )

    else:

        print(
            "[5] No email column in this table"
        )


    # ============================================================
    # STEP 6 — VALIDATE NUMERIC VALUES
    # ============================================================
    #
    # Invalid negative values are removed from the cleaned data.
    #
    # Rules:
    #
    #   products.price       >= 0
    #   order_details.qty   >= 0
    #   order_details.price >= 0
    #   order_details.discount >= 0
    #   payments.amount     >= 0
    #
    # ============================================================

    rows_removed_numeric = 0


    if table_name == "products":

        if has_column(df, "price"):

            invalid_count = (
                df.filter(
                    F.col("price") < 0
                ).count()
            )

            rows_removed_numeric += invalid_count

            df = df.filter(
                F.col("price").isNull()
                |
                (F.col("price") >= 0)
            )

            print(
                f"[6] Negative product prices removed: "
                f"{invalid_count}"
            )


    if table_name == "order_details":

        if has_column(df, "quantity"):

            invalid_count = (
                df.filter(
                    F.col("quantity") < 0
                ).count()
            )

            rows_removed_numeric += invalid_count

            df = df.filter(
                F.col("quantity").isNull()
                |
                (F.col("quantity") >= 0)
            )

            print(
                f"[6] Negative quantities removed: "
                f"{invalid_count}"
            )


        if has_column(df, "unit_price"):

            invalid_count = (
                df.filter(
                    F.col("unit_price") < 0
                ).count()
            )

            rows_removed_numeric += invalid_count

            df = df.filter(
                F.col("unit_price").isNull()
                |
                (F.col("unit_price") >= 0)
            )

            print(
                f"[6] Negative unit prices removed: "
                f"{invalid_count}"
            )


        if has_column(df, "discount"):

            invalid_count = (
                df.filter(
                    F.col("discount") < 0
                ).count()
            )

            rows_removed_numeric += invalid_count

            df = df.filter(
                F.col("discount").isNull()
                |
                (F.col("discount") >= 0)
            )

            print(
                f"[6] Negative discounts removed: "
                f"{invalid_count}"
            )


    if table_name == "payments":

        if has_column(df, "amount"):

            invalid_count = (
                df.filter(
                    F.col("amount") < 0
                ).count()
            )

            rows_removed_numeric += invalid_count

            df = df.filter(
                F.col("amount").isNull()
                |
                (F.col("amount") >= 0)
            )

            print(
                f"[6] Negative payment amounts removed: "
                f"{invalid_count}"
            )


    # ============================================================
    # STEP 7 — VALIDATE DATES
    # ============================================================
    #
    # Invalid date strings become NULL after to_date().
    #
    # For shipments:
    #
    #   delivery_date < ship_date
    #
    # is treated as invalid and delivery_date becomes NULL.
    #
    # We do not invent replacement dates.
    #
    # ============================================================

    date_columns = []

    for field in df.schema.fields:

        if isinstance(
            field.dataType,
            DateType
        ):

            date_columns.append(
                field.name
            )


    print(
        f"[7] Date columns validated: "
        f"{date_columns}"
    )


    # ------------------------------------------------------------
    # Shipment date-order validation
    # ------------------------------------------------------------

    if (
        table_name == "shipments"
        and
        has_column(df, "ship_date")
        and
        has_column(df, "delivery_date")
    ):

        invalid_delivery_count = (
            df.filter(
                F.col("ship_date").isNotNull()
                &
                F.col("delivery_date").isNotNull()
                &
                (
                    F.col("delivery_date")
                    <
                    F.col("ship_date")
                )
            ).count()
        )

        df = df.withColumn(
            "delivery_date",
            F.when(
                F.col("ship_date").isNotNull()
                &
                F.col("delivery_date").isNotNull()
                &
                (
                    F.col("delivery_date")
                    <
                    F.col("ship_date")
                ),
                None
            ).otherwise(
                F.col("delivery_date")
            )
        )

        print(
            "[7] Invalid delivery-before-shipment "
            f"records corrected: {invalid_delivery_count}"
        )


    # ============================================================
    # STEP 8 — HANDLE NULL VALUES
    # ============================================================
    #
    # Rules:
    #
    #   Business-key NULL:
    #       Remove the record because it cannot be uniquely
    #       identified.
    #
    #   String NULL:
    #       "UNKNOWN"
    #
    #   Numeric NULL:
    #       0
    #
    #   Date NULL:
    #       Keep NULL because a reliable date cannot be invented.
    #
    # ============================================================

    rows_before_null_handling = df.count()


    # ------------------------------------------------------------
    # Remove rows with NULL business keys
    # ------------------------------------------------------------

    table_keys = cleaning_keys[table_name]

    valid_key_condition = None

    for key_column in table_keys:

        if has_column(df, key_column):

            condition = F.col(
                key_column
            ).isNotNull()

            if valid_key_condition is None:

                valid_key_condition = condition

            else:

                valid_key_condition = (
                    valid_key_condition
                    &
                    condition
                )


    if valid_key_condition is not None:

        df = df.filter(
            valid_key_condition
        )


    rows_after_key_null_removal = df.count()

    key_null_removed = (
        rows_before_null_handling
        -
        rows_after_key_null_removal
    )


    # ------------------------------------------------------------
    # Fill remaining string NULLs with UNKNOWN
    # ------------------------------------------------------------

    string_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            StringType
        )
    ]

    if string_columns:

        df = df.fillna(
            "UNKNOWN",
            subset=string_columns
        )


    # ------------------------------------------------------------
    # Fill numeric NULLs with 0
    # ------------------------------------------------------------

    numeric_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            (
                IntegerType,
                LongType,
                DoubleType
            )
        )
    ]

    if numeric_columns:

        df = df.fillna(
            0,
            subset=numeric_columns
        )


    print(
        f"[8] NULL handling completed. "
        f"Rows removed because of NULL business keys: "
        f"{key_null_removed}"
    )


    # ============================================================
    # STEP 9 — REMOVE DUPLICATES
    # ============================================================
    #
    # One row is kept for each business key.
    #
    # product_suppliers uses:
    #
    #   product_id + supplier_id
    #
    # ============================================================

    rows_before_duplicates = df.count()

    existing_keys = [
        key
        for key in table_keys
        if has_column(df, key)
    ]

    if existing_keys:

        df = df.dropDuplicates(
            existing_keys
        )

    rows_after_duplicates = df.count()

    duplicates_removed = (
        rows_before_duplicates
        -
        rows_after_duplicates
    )


    print(
        f"[9] Duplicate records removed: "
        f"{duplicates_removed}"
    )


    # ============================================================
    # STEP 10 — FINAL CLEANED DATAFRAME
    # ============================================================

    final_count = df.count()

    cleaned_dfs[table_name] = df


    # ============================================================
    # STEP 11 — SAVE CLEANED DATA
    # ============================================================
    #
    # Parquet is used because it is columnar, compressed, and
    # suitable for downstream Spark/Redshift processing.
    #
    # Each source table gets its own folder.
    #
    # ============================================================

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )


    (
        df.write
        .mode("overwrite")
        .parquet(output_path)
    )


    print(
        f"[10] Saved cleaned data:"
    )

    print(
        f"     {output_path}"
    )

    print(
        f"     Final rows: {final_count}"
    )


    # ============================================================
    # SAVE SUMMARY
    # ============================================================

    cleaning_summary.append(
        (
            table_name,
            original_count,
            final_count,
            original_columns,
            len(df.columns),
            key_null_removed,
            duplicates_removed,
            rows_removed_numeric
        )
    )


# ================================================================
# 10. CLEANING SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("PART 3 — CLEANING SUMMARY")
print("=" * 80)


cleaning_summary_df = spark.createDataFrame(
    cleaning_summary,
    [
        "table",
        "original_rows",
        "final_rows",
        "original_columns",
        "final_columns",
        "null_key_rows_removed",
        "duplicate_rows_removed",
        "invalid_numeric_rows_removed"
    ]
)


cleaning_summary_df = (
    cleaning_summary_df
    .orderBy("table")
)


cleaning_summary_df.show(
    truncate=False
)


# ================================================================
# 11. DISPLAY CLEANED SCHEMAS
# ================================================================

print("\n" + "=" * 80)
print("CLEANED DATA SCHEMAS")
print("=" * 80)


for table_name, df in cleaned_dfs.items():

    print("\n" + "-" * 80)
    print(f"CLEANED SCHEMA: {table_name}")
    print("-" * 80)

    df.printSchema()


# ================================================================
# 12. VERIFY SAVED DATA
# ================================================================
#
# Read the Parquet outputs back from S3 and verify that they
# actually exist and contain data.
#
# ================================================================

print("\n" + "=" * 80)
print("VERIFYING PROCESSED PARQUET OUTPUT")
print("=" * 80)


verification_results = []


for table_name in cleaning_tables:

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )

    try:

        verification_df = spark.read.parquet(
            output_path
        )

        verification_count = (
            verification_df.count()
        )

        verification_results.append(
            (
                table_name,
                "SUCCESS",
                verification_count
            )
        )

        print(
            f"[OK] {table_name}: "
            f"{verification_count} rows"
        )

    except Exception as e:

        verification_results.append(
            (
                table_name,
                "FAILED",
                0
            )
        )

        print(
            f"[FAILED] {table_name}"
        )

        print(str(e))


# ================================================================
# 13. FINAL VERIFICATION TABLE
# ================================================================

verification_df = spark.createDataFrame(
    verification_results,
    [
        "table",
        "status",
        "saved_row_count"
    ]
)


verification_df.show(
    truncate=False
)


# ================================================================
# 14. FINAL MESSAGE
# ================================================================

print("\n" + "=" * 80)
print("PART 3 — DATA CLEANING COMPLETED")
print("=" * 80)

print("""
Cleaning performed:

[OK] Column names standardized to snake_case
[OK] String whitespace removed
[OK] Empty strings converted to NULL
[OK] Status values standardized
[OK] Email addresses cleaned and validated
[OK] Data types converted
[OK] Invalid negative numeric values removed
[OK] Invalid dates converted to NULL
[OK] Invalid shipment date ordering handled
[OK] NULL business keys removed
[OK] Remaining string NULLs replaced with UNKNOWN
[OK] Remaining numeric NULLs replaced with 0
[OK] Duplicate business keys removed
[OK] Cleaned datasets saved as Parquet
[OK] Saved Parquet outputs verified

orders_json is intentionally reserved for Part 6 because it
requires nested JSON parsing and order_details explosion.
""")

print(f"Processed location: {PROCESSED_PATH}")

print("=" * 80)

PART 3 — DATA CLEANING
Input datasets available: 13
Output path: s3://aws-ecommerce-s3/ecommerce/processed

CLEANING TABLE: customers
Original rows    : 10000
Original columns : 7

[1] Column names standardized
[2] Unnecessary spaces removed
[3] String values standardized
[4] Data types converted
[5] Email addresses trimmed, lowercased, and validated
[7] Date columns validated: ['registration_date']
[8] NULL handling completed. Rows removed because of NULL business keys: 0
[9] Duplicate records removed: 0
[10] Saved cleaned data:
     s3://aws-ecommerce-s3/ecommerce/processed/customers
     Final rows: 10000

CLEANING TABLE: categories
Original rows    : 20
Original columns : 2

[1] Column names standardized
[2] Unnecessary spaces removed
[3] String values standardized
[4] Data types converted
[5] No email column in this table
[7] Date columns validated: []
[8] NULL handling completed. Rows removed because of NULL business keys: 0
[9] Duplicate records removed: 0
[10] Saved cleaned dat

In [11]:
# ============================================================
# PART 4 — BUILD DIMENSIONS
# Tasks 6–15
#
# This cell builds the dimensional layer from the cleaned
# DataFrames created in Part 3: `cleaned_dfs`.
#
# Dimensions created:
#   1. dim_customer
#   2. dim_category
#   3. dim_department
#   4. dim_product
#   5. dim_supplier
#   6. dim_employee
#   7. dim_shipper
#   8. dim_payment_method
#   9. dim_order_status
#  10. dim_date
#
# Design:
#   - Original business/source IDs are preserved.
#   - Surrogate keys are generated for dimensions.
#   - Dimension grains are explicitly defined.
#   - Duplicate business records are removed.
#   - Null/default values are handled.
#   - Product is enriched using category + department.
#   - dim_date covers the complete period of source data.
#   - All dimensions are stored as Parquet.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    DateType
)

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

PROCESSED_PATH = "s3://aws-ecommerce-s3/ecommerce/processed"

print("=" * 80)
print("PART 4 — BUILD DIMENSIONS")
print("=" * 80)

print("\nSource DataFrames available from Part 3:")
for table_name in sorted(cleaned_dfs.keys()):
    print(f"  - {table_name}")

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def write_dimension(df, dimension_name):
    """
    Write a dimension as Parquet and verify the written result.
    """

    output_path = f"{PROCESSED_PATH}/{dimension_name}"

    print("\n" + "-" * 80)
    print(f"Writing {dimension_name}")
    print(f"Path: {output_path}")
    print("-" * 80)

    df.write.mode("overwrite").parquet(output_path)

    # Read back for verification
    verification_df = spark.read.parquet(output_path)

    print(f"Rows written: {verification_df.count()}")
    print(f"Columns: {len(verification_df.columns)}")

    verification_df.printSchema()
    verification_df.show(5, truncate=False)

    return verification_df


# ============================================================
# TASK 6 — dim_customer
# ============================================================
#
# Grain:
#   One row per unique customer.
#
# Source:
#   customers
#
# Attributes:
#   customer_key       -> surrogate key
#   customer_id        -> original business/source key
#   first_name
#   last_name
#   email
#   phone              -> included if available in source
#   city
#   country
#   registration_date
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 6 — dim_customer")
print("=" * 80)

customers = cleaned_dfs["customers"]

print("\nOriginal customers schema:")
customers.printSchema()

# Remove duplicate customers based on the original business key.
customer_base = (
    customers
    .dropDuplicates(["customer_id"])
)

# Some versions of the source data may not contain phone.
# Add it as NULL if it is unavailable so the dimension still
# follows the required dimensional structure.
if "phone" not in customer_base.columns:
    customer_base = customer_base.withColumn(
        "phone",
        F.lit(None).cast(StringType())
    )

# Deterministic surrogate key.
customer_window = Window.orderBy("customer_id")

dim_customer = (
    customer_base
    .withColumn(
        "customer_key",
        F.row_number().over(customer_window).cast(LongType())
    )
    .select(
        "customer_key",
        "customer_id",
        F.coalesce(F.col("first_name"), F.lit("UNKNOWN")).alias("first_name"),
        F.coalesce(F.col("last_name"), F.lit("UNKNOWN")).alias("last_name"),
        F.coalesce(F.col("email"), F.lit("UNKNOWN")).alias("email"),
        F.coalesce(F.col("phone"), F.lit("UNKNOWN")).alias("phone"),
        F.coalesce(F.col("city"), F.lit("UNKNOWN")).alias("city"),
        F.coalesce(F.col("country"), F.lit("UNKNOWN")).alias("country"),
        "registration_date"
    )
)

print("\nCustomer dimension:")
dim_customer.show(10, truncate=False)

print(f"dim_customer row count: {dim_customer.count()}")

dim_customer = write_dimension(
    dim_customer,
    "dim_customer"
)


# ============================================================
# TASK 7 — dim_category
# ============================================================
#
# Grain:
#   One row per category.
#
# Source:
#   categories
#
# Attributes:
#   category_key
#   category_id
#   category_name
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 7 — dim_category")
print("=" * 80)

categories = cleaned_dfs["categories"]

category_base = (
    categories
    .dropDuplicates(["category_id"])
)

category_window = Window.orderBy("category_id")

dim_category = (
    category_base
    .withColumn(
        "category_key",
        F.row_number().over(category_window).cast(LongType())
    )
    .select(
        "category_key",
        "category_id",
        F.coalesce(
            F.col("category_name"),
            F.lit("UNKNOWN")
        ).alias("category_name")
    )
)

print("\nCategory dimension:")
dim_category.show(10, truncate=False)

print(f"dim_category row count: {dim_category.count()}")

dim_category = write_dimension(
    dim_category,
    "dim_category"
)


# ============================================================
# TASK 8 — dim_department
# ============================================================
#
# Grain:
#   One row per department.
#
# Source:
#   departments
#
# Attributes:
#   department_key
#   department_id
#   department_name
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 8 — dim_department")
print("=" * 80)

departments = cleaned_dfs["departments"]

department_base = (
    departments
    .dropDuplicates(["department_id"])
)

department_window = Window.orderBy("department_id")

dim_department = (
    department_base
    .withColumn(
        "department_key",
        F.row_number().over(department_window).cast(LongType())
    )
    .select(
        "department_key",
        "department_id",
        F.coalesce(
            F.col("department_name"),
            F.lit("UNKNOWN")
        ).alias("department_name")
    )
)

print("\nDepartment dimension:")
dim_department.show(10, truncate=False)

print(f"dim_department row count: {dim_department.count()}")

dim_department = write_dimension(
    dim_department,
    "dim_department"
)


# ============================================================
# TASK 9 — dim_product
# ============================================================
#
# Grain:
#   One row per unique product.
#
# Source:
#   products
#   categories
#
# Actual products schema:
#   product_id
#   category_id
#   product_name
#   brand
#   price
#   cost
#   stock
#
# Important:
#   The actual products source does NOT contain department_id.
#   Therefore, there is no valid direct products -> departments
#   relationship to join here.
#
#   We preserve the category relationship and include the
#   available product-level business attributes.
#
#   Product cost is available as `cost`, so we use it directly.
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 9 — dim_product")
print("=" * 80)

products = cleaned_dfs["products"]

print("\nActual product columns:")
print(products.columns)

print("\nProduct source schema:")
products.printSchema()

# ------------------------------------------------------------
# Remove duplicate products using the business key.
# ------------------------------------------------------------

product_base = (
    products
    .dropDuplicates(["product_id"])
)

# ------------------------------------------------------------
# Category lookup
#
# dim_category was already created in Task 7.
# ------------------------------------------------------------

category_lookup = (
    dim_category
    .select(
        "category_key",
        "category_id",
        "category_name"
    )
)

# ------------------------------------------------------------
# Join products with categories.
# ------------------------------------------------------------

product_enriched = (
    product_base.alias("p")
    .join(
        category_lookup.alias("c"),
        F.col("p.category_id") == F.col("c.category_id"),
        "left"
    )
)

# ------------------------------------------------------------
# Generate product surrogate key.
# ------------------------------------------------------------

product_window = Window.orderBy(
    F.col("p.product_id")
)

dim_product = (
    product_enriched

    .withColumn(
        "product_key",
        F.row_number()
        .over(product_window)
        .cast(LongType())
    )

    .select(

        # Surrogate key
        "product_key",

        # Original business key
        F.col("p.product_id").alias("product_id"),

        # Category relationship
        F.col("c.category_key").alias("category_key"),
        F.col("p.category_id").alias("category_id"),

        F.coalesce(
            F.col("c.category_name"),
            F.lit("UNKNOWN")
        ).alias("category_name"),

        # Product attributes
        F.coalesce(
            F.col("p.product_name"),
            F.lit("UNKNOWN")
        ).alias("product_name"),

        F.coalesce(
            F.col("p.brand"),
            F.lit("UNKNOWN")
        ).alias("brand"),

        F.col("p.price")
            .cast(DoubleType())
            .alias("price"),

        # Actual source column is `cost`
        F.col("p.cost")
            .cast(DoubleType())
            .alias("product_cost"),

        F.col("p.stock")
            .cast(IntegerType())
            .alias("stock")
    )
)

# ------------------------------------------------------------
# Product-level business attributes
#
# profit_amount:
#     price - cost
#
# profit_margin_percentage:
#     ((price - cost) / price) * 100
#
# Only calculate margin when price is not zero.
# ------------------------------------------------------------

dim_product = (
    dim_product

    .withColumn(
        "profit_amount",
        F.when(
            F.col("product_cost").isNotNull() &
            F.col("price").isNotNull(),
            F.col("price") - F.col("product_cost")
        ).otherwise(
            F.lit(None).cast(DoubleType())
        )
    )

    .withColumn(
        "profit_margin_percentage",
        F.when(
            F.col("price").isNotNull() &
            (F.col("price") != 0) &
            F.col("product_cost").isNotNull(),
            (
                (
                    F.col("price") -
                    F.col("product_cost")
                )
                / F.col("price")
            ) * 100
        ).otherwise(
            F.lit(None).cast(DoubleType())
        )
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nProduct dimension schema:")
dim_product.printSchema()

print("\nSample product dimension records:")
dim_product.show(
    10,
    truncate=False
)

print("\nProduct dimension row count:")
print(dim_product.count())

print("\nProduct dimension column count:")
print(len(dim_product.columns))

print("\nChecking duplicate product IDs:")

duplicate_products = (
    dim_product
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_products.count()

print(
    f"Duplicate product IDs: {duplicate_count}"
)

if duplicate_count == 0:
    print("✓ No duplicate product IDs found.")
else:
    print("WARNING: Duplicate product IDs found.")
    duplicate_products.show(20)

# ------------------------------------------------------------
# Check category relationships.
# ------------------------------------------------------------

print("\nChecking products without a category match:")

unmatched_categories = (
    dim_product
    .filter(
        F.col("category_key").isNull()
        &
        F.col("category_id").isNotNull()
    )
)

unmatched_category_count = unmatched_categories.count()

print(
    f"Products with unmatched categories: "
    f"{unmatched_category_count}"
)

if unmatched_category_count > 0:
    unmatched_categories.select(
        "product_id",
        "category_id"
    ).show(20)

# ------------------------------------------------------------
# Write dimension
# ------------------------------------------------------------

dim_product = write_dimension(
    dim_product,
    "dim_product"
)


# ============================================================
# TASK 10 — dim_supplier
# ============================================================
#
# Grain:
#   One row per unique supplier.
#
# Source:
#   suppliers
#
# Includes appropriate supplier information and surrogate key.
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 10 — dim_supplier")
print("=" * 80)

suppliers = cleaned_dfs["suppliers"]

supplier_base = (
    suppliers
    .dropDuplicates(["supplier_id"])
)

supplier_window = Window.orderBy("supplier_id")

# Dynamically select available supplier attributes.
supplier_select = [
    "supplier_key",
    "supplier_id"
]

if "company_name" in supplier_base.columns:
    supplier_select.append(
        F.coalesce(
            F.col("company_name"),
            F.lit("UNKNOWN")
        ).alias("company_name")
    )

if "contact_name" in supplier_base.columns:
    supplier_select.append(
        F.coalesce(
            F.col("contact_name"),
            F.lit("UNKNOWN")
        ).alias("contact_name")
    )

if "email" in supplier_base.columns:
    supplier_select.append(
        F.coalesce(
            F.col("email"),
            F.lit("UNKNOWN")
        ).alias("email")
    )

dim_supplier = (
    supplier_base
    .withColumn(
        "supplier_key",
        F.row_number().over(supplier_window).cast(LongType())
    )
    .select(*supplier_select)
)

print("\nSupplier dimension:")
dim_supplier.show(10, truncate=False)

print(f"dim_supplier row count: {dim_supplier.count()}")

dim_supplier = write_dimension(
    dim_supplier,
    "dim_supplier"
)


# ============================================================
# TASK 11 — dim_employee
# ============================================================
#
# Grain:
#   One row per unique employee.
#
# Actual source columns:
#   employee_id
#   manager_id
#   department_id
#   first_name
#   last_name
#   salary
#   hire_date
#
# The source does NOT contain email, so email is not included.
#
# The employee dimension is also connected to dim_department
# using department_id.
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 11 — dim_employee")
print("=" * 80)

employees = cleaned_dfs["employees"]

print("\nActual employee columns:")
print(employees.columns)

print("\nEmployee source schema:")
employees.printSchema()

# ------------------------------------------------------------
# Remove duplicate employees using the business key.
# ------------------------------------------------------------

employee_base = (
    employees
    .dropDuplicates(["employee_id"])
)

# ------------------------------------------------------------
# Department lookup
#
# dim_department was created in Task 8.
# ------------------------------------------------------------

department_lookup = (
    dim_department
    .select(
        "department_key",
        "department_id"
    )
)

# ------------------------------------------------------------
# Connect employees to departments.
# ------------------------------------------------------------

employee_enriched = (
    employee_base.alias("e")
    .join(
        department_lookup.alias("d"),
        F.col("e.department_id") == F.col("d.department_id"),
        "left"
    )
)

# ------------------------------------------------------------
# Generate employee surrogate key.
# ------------------------------------------------------------

employee_window = Window.orderBy(
    F.col("e.employee_id")
)

dim_employee = (
    employee_enriched

    .withColumn(
        "employee_key",
        F.row_number()
        .over(employee_window)
        .cast(LongType())
    )

    .select(

        # Surrogate key
        "employee_key",

        # Original employee business key
        F.col("e.employee_id").alias("employee_id"),

        # Manager relationship
        F.col("e.manager_id").alias("manager_id"),

        # Department relationship
        F.col("d.department_key").alias("department_key"),
        F.col("e.department_id").alias("department_id"),

        # Employee information
        F.coalesce(
            F.col("e.first_name"),
            F.lit("UNKNOWN")
        ).alias("first_name"),

        F.coalesce(
            F.col("e.last_name"),
            F.lit("UNKNOWN")
        ).alias("last_name"),

        # Financial / employment attributes
        F.col("e.salary")
            .cast(DoubleType())
            .alias("salary"),

        F.col("e.hire_date")
            .cast(DateType())
            .alias("hire_date")
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nEmployee dimension schema:")
dim_employee.printSchema()

print("\nSample employee dimension records:")
dim_employee.show(
    10,
    truncate=False
)

print("\nEmployee dimension row count:")
print(dim_employee.count())

print("\nEmployee dimension column count:")
print(len(dim_employee.columns))

# ------------------------------------------------------------
# Check duplicate employee IDs
# ------------------------------------------------------------

print("\nChecking duplicate employee IDs:")

duplicate_employees = (
    dim_employee
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_employee_count = duplicate_employees.count()

print(
    f"Duplicate employee IDs: "
    f"{duplicate_employee_count}"
)

if duplicate_employee_count == 0:
    print("✓ No duplicate employee IDs found.")
else:
    print("WARNING: Duplicate employee IDs found.")
    duplicate_employees.show(20)

# ------------------------------------------------------------
# Check department relationships
# ------------------------------------------------------------

print("\nChecking employees without a department match:")

unmatched_departments = (
    dim_employee
    .filter(
        F.col("department_key").isNull()
        &
        F.col("department_id").isNotNull()
    )
)

unmatched_department_count = unmatched_departments.count()

print(
    f"Employees with unmatched departments: "
    f"{unmatched_department_count}"
)

if unmatched_department_count > 0:
    unmatched_departments.select(
        "employee_id",
        "department_id"
    ).show(20)

# ------------------------------------------------------------
# Write dimension
# ------------------------------------------------------------

dim_employee = write_dimension(
    dim_employee,
    "dim_employee"
)


# ============================================================
# TASK 12 — dim_shipper
# ============================================================
#
# Grain:
#   One row per unique shipper.
#
# Source:
#   shippers
#
# The historical source uses `company_name` after the
# standardization performed in Part 3.
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 12 — dim_shipper")
print("=" * 80)

shippers = cleaned_dfs["shippers"]

shipper_base = (
    shippers
    .dropDuplicates(["shipper_id"])
)

shipper_window = Window.orderBy("shipper_id")

dim_shipper = (
    shipper_base
    .withColumn(
        "shipper_key",
        F.row_number().over(shipper_window).cast(LongType())
    )
    .select(
        "shipper_key",
        "shipper_id",
        F.coalesce(
            F.col("company_name"),
            F.lit("UNKNOWN")
        ).alias("company_name")
    )
)

print("\nShipper dimension:")
dim_shipper.show(10, truncate=False)

print(f"dim_shipper row count: {dim_shipper.count()}")

dim_shipper = write_dimension(
    dim_shipper,
    "dim_shipper"
)


# ============================================================
# TASK 13 — dim_payment_method
# ============================================================
#
# Grain:
#   One row per unique payment method.
#
# Source:
#   payments
#
# Attributes:
#   payment_method_key
#   payment_method
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 13 — dim_payment_method")
print("=" * 80)

payments = cleaned_dfs["payments"]

payment_methods = (
    payments
    .select("payment_method")
    .filter(F.col("payment_method").isNotNull())
    .withColumn(
        "payment_method",
        F.trim(F.col("payment_method"))
    )
    .filter(F.col("payment_method") != "")
    .dropDuplicates(["payment_method"])
)

payment_method_window = Window.orderBy("payment_method")

dim_payment_method = (
    payment_methods
    .withColumn(
        "payment_method_key",
        F.row_number().over(payment_method_window).cast(LongType())
    )
    .select(
        "payment_method_key",
        F.col("payment_method")
    )
)

print("\nPayment method dimension:")
dim_payment_method.show(100, truncate=False)

print(
    f"dim_payment_method row count: "
    f"{dim_payment_method.count()}"
)

dim_payment_method = write_dimension(
    dim_payment_method,
    "dim_payment_method"
)


# ============================================================
# TASK 14 — dim_order_status
# ============================================================
#
# Grain:
#   One row per unique order status.
#
# Source:
#   orders
#
# Attributes:
#   order_status_key
#   order_status
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 14 — dim_order_status")
print("=" * 80)

orders = cleaned_dfs["orders"]

order_statuses = (
    orders
    .select("status")
    .filter(F.col("status").isNotNull())
    .withColumn(
        "order_status",
        F.trim(F.col("status"))
    )
    .filter(F.col("order_status") != "")
    .select("order_status")
    .dropDuplicates(["order_status"])
)

order_status_window = Window.orderBy("order_status")

dim_order_status = (
    order_statuses
    .withColumn(
        "order_status_key",
        F.row_number().over(order_status_window).cast(LongType())
    )
    .select(
        "order_status_key",
        "order_status"
    )
)

print("\nOrder status dimension:")
dim_order_status.show(100, truncate=False)

print(
    f"dim_order_status row count: "
    f"{dim_order_status.count()}"
)

dim_order_status = write_dimension(
    dim_order_status,
    "dim_order_status"
)


# ============================================================
# TASK 15 — dim_date
# ============================================================
#
# Grain:
#   One row per calendar date.
#
# Attributes:
#   date_key
#   full_date
#   day
#   month
#   month_name
#   quarter
#   year
#   week
#   day_name
#   is_weekend
#
# The date dimension covers the complete period represented
# by the source data.
#
# We collect dates from all relevant historical date columns:
#   customers.registration_date
#   orders.order_date
#   payments.payment_date
#   shipments.ship_date
#   shipments.delivery_date
#
# If one of these columns is absent, it is skipped.
# ============================================================

print("\n\n" + "=" * 80)
print("TASK 15 — dim_date")
print("=" * 80)

date_sources = []

# ------------------------------------------------------------
# Customer registration dates
# ------------------------------------------------------------

if "registration_date" in customers.columns:
    date_sources.append(
        customers
        .select(
            F.col("registration_date").cast(DateType()).alias("date_value")
        )
        .filter(F.col("date_value").isNotNull())
    )

# ------------------------------------------------------------
# Order dates
# ------------------------------------------------------------

if "order_date" in orders.columns:
    date_sources.append(
        orders
        .select(
            F.col("order_date").cast(DateType()).alias("date_value")
        )
        .filter(F.col("date_value").isNotNull())
    )

# ------------------------------------------------------------
# Payment dates
# ------------------------------------------------------------

if "payment_date" in payments.columns:
    date_sources.append(
        payments
        .select(
            F.col("payment_date").cast(DateType()).alias("date_value")
        )
        .filter(F.col("date_value").isNotNull())
    )

# ------------------------------------------------------------
# Shipment dates
# ------------------------------------------------------------

shipments = cleaned_dfs["shipments"]

if "ship_date" in shipments.columns:
    date_sources.append(
        shipments
        .select(
            F.col("ship_date").cast(DateType()).alias("date_value")
        )
        .filter(F.col("date_value").isNotNull())
    )

if "delivery_date" in shipments.columns:
    date_sources.append(
        shipments
        .select(
            F.col("delivery_date").cast(DateType()).alias("date_value")
        )
        .filter(F.col("date_value").isNotNull())
    )

# ------------------------------------------------------------
# Combine all dates
# ------------------------------------------------------------

if len(date_sources) == 0:
    raise ValueError(
        "No valid date columns were found to build dim_date."
    )

all_source_dates = date_sources[0]

for date_df in date_sources[1:]:
    all_source_dates = all_source_dates.unionByName(date_df)

all_source_dates = all_source_dates.dropDuplicates()

# Determine complete source period.
date_range = (
    all_source_dates
    .agg(
        F.min("date_value").alias("min_date"),
        F.max("date_value").alias("max_date")
    )
    .collect()[0]
)

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print(f"\nMinimum source date: {min_date}")
print(f"Maximum source date: {max_date}")

if min_date is None or max_date is None:
    raise ValueError(
        "Unable to determine the date range for dim_date."
    )

# ------------------------------------------------------------
# Generate every calendar date between min_date and max_date.
# ------------------------------------------------------------

date_sequence = (
    spark
    .range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(min_date).cast(DateType()),
                F.lit(max_date).cast(DateType()),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
)

# ------------------------------------------------------------
# Build date attributes.
# ------------------------------------------------------------

dim_date = (
    date_sequence

    # YYYYMMDD integer surrogate/date key.
    .withColumn(
        "date_key",
        F.date_format(
            F.col("full_date"),
            "yyyyMMdd"
        ).cast(IntegerType())
    )

    .withColumn(
        "day",
        F.dayofmonth("full_date")
    )

    .withColumn(
        "month",
        F.month("full_date")
    )

    .withColumn(
        "month_name",
        F.date_format("full_date", "MMMM")
    )

    .withColumn(
        "quarter",
        F.quarter("full_date")
    )

    .withColumn(
        "year",
        F.year("full_date")
    )

    .withColumn(
        "week",
        F.weekofyear("full_date")
    )

    .withColumn(
        "day_name",
        F.date_format("full_date", "EEEE")
    )

    # Spark dayofweek:
    # 1 = Sunday
    # 7 = Saturday
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin([1, 7])
    )

    .select(
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week",
        "day_name",
        "is_weekend"
    )

    .orderBy("full_date")
)

print("\nDate dimension:")
dim_date.show(10, truncate=False)

print("\nLast dates:")
dim_date.orderBy(
    F.col("full_date").desc()
).show(10, truncate=False)

print(f"dim_date row count: {dim_date.count()}")

dim_date.printSchema()

dim_date = write_dimension(
    dim_date,
    "dim_date"
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n\n" + "=" * 80)
print("PART 4 — FINAL DIMENSION VERIFICATION")
print("=" * 80)

dimension_dfs = {
    "dim_customer": dim_customer,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_product": dim_product,
    "dim_supplier": dim_supplier,
    "dim_employee": dim_employee,
    "dim_shipper": dim_shipper,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status,
    "dim_date": dim_date
}

dimension_summary = []

for dimension_name, dimension_df in dimension_dfs.items():

    row_count = dimension_df.count()
    column_count = len(dimension_df.columns)

    dimension_summary.append(
        (
            dimension_name,
            row_count,
            column_count
        )
    )

dimension_summary_df = spark.createDataFrame(
    dimension_summary,
    [
        "dimension_name",
        "row_count",
        "column_count"
    ]
)

dimension_summary_df = dimension_summary_df.orderBy(
    "dimension_name"
)

dimension_summary_df.show(
    100,
    truncate=False
)

print("\n" + "=" * 80)
print("DIMENSION OUTPUT LOCATIONS")
print("=" * 80)

for dimension_name in dimension_dfs.keys():
    print(
        f"{dimension_name}: "
        f"{PROCESSED_PATH}/{dimension_name}"
    )

print("\n" + "=" * 80)
print("PART 4 COMPLETED SUCCESSFULLY")
print("=" * 80)

PART 4 — BUILD DIMENSIONS

Source DataFrames available from Part 3:
  - categories
  - customers
  - departments
  - employees
  - order_details
  - orders
  - payments
  - product_suppliers
  - products
  - shipments
  - shippers
  - suppliers


TASK 6 — dim_customer

Original customers schema:
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = false)
 |-- last_name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- city: string (nullable = false)
 |-- country: string (nullable = false)
 |-- registration_date: date (nullable = true)


Customer dimension:
+------------+-----------+----------+---------+-----------------------------+-------+----------+--------------+-----------------+
|customer_key|customer_id|first_name|last_name|email                        |phone  |city      |country       |registration_date|
+------------+-----------+----------+---------+-----------------------------+-------+----------+--------------+--------------

In [13]:
# ============================================================
# PART 5 — BUILD FACT TABLES
# Tasks 16–21
#
# Fact tables:
#   1. fact_order
#   2. fact_order_detail
#   3. fact_payment
#   4. fact_shipment
#   5. fact_customer_sales
#   6. fact_product_sales
#
# IMPORTANT DESIGN DECISIONS
# ------------------------------------------------------------
# All facts preserve the original business IDs while also
# using dimension surrogate keys wherever applicable.
#
# Discount assumption:
#   The source `discount` column is treated as a percentage.
#   Example:
#       quantity = 2
#       unit_price = 100
#       discount = 10
#
#       gross_sales = 200
#       discount_amount = 20
#       net_sales = 180
#
# This can be changed by setting:
#       DISCOUNT_IS_PERCENT = False
#
# Grain:
#
# fact_order:
#   One row per order.
#
# fact_order_detail:
#   One row per order-detail record / product line within an order.
#
# fact_payment:
#   One row per payment transaction.
#
# fact_shipment:
#   One row per shipment.
#
# fact_customer_sales:
#   One row per customer per day.
#
# fact_product_sales:
#   One row per product per day.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    DateType
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PROCESSED_PATH = "s3://aws-ecommerce-s3/ecommerce/processed"

# Based on the source data structure, Discount is treated
# as a percentage.
DISCOUNT_IS_PERCENT = True

print("=" * 90)
print("PART 5 — BUILD FACT TABLES")
print("=" * 90)


# ============================================================
# 0. LOAD / REFERENCE CLEANED SOURCE DATA
# ============================================================

customers = cleaned_dfs["customers"]
orders = cleaned_dfs["orders"]
order_details = cleaned_dfs["order_details"]
products = cleaned_dfs["products"]
payments = cleaned_dfs["payments"]
shipments = cleaned_dfs["shipments"]
shippers = cleaned_dfs["shippers"]

print("\nSource DataFrames:")
print(f"customers       : {customers.count():,}")
print(f"orders          : {orders.count():,}")
print(f"order_details   : {order_details.count():,}")
print(f"products        : {products.count():,}")
print(f"payments        : {payments.count():,}")
print(f"shipments       : {shipments.count():,}")
print(f"shippers        : {shippers.count():,}")


# ============================================================
# 1. HELPER FUNCTION — WRITE + VERIFY FACT
# ============================================================

def write_fact(df, fact_name):

    output_path = f"{PROCESSED_PATH}/{fact_name}"

    print("\n" + "-" * 90)
    print(f"Writing {fact_name}")
    print(f"Path: {output_path}")
    print("-" * 90)

    df.write.mode("overwrite").parquet(output_path)

    # Read back to verify the actual Parquet output.
    verification_df = spark.read.parquet(output_path)

    print(f"Rows written   : {verification_df.count():,}")
    print(f"Columns        : {len(verification_df.columns)}")

    print("\nSchema:")
    verification_df.printSchema()

    print("\nSample records:")
    verification_df.show(5, truncate=False)

    return verification_df


# ============================================================
# 2. CREATE REUSABLE DIMENSION LOOKUPS
# ============================================================

print("\n" + "=" * 90)
print("PREPARING DIMENSION LOOKUPS")
print("=" * 90)

# Customer
customer_lookup = (
    dim_customer
    .select(
        "customer_key",
        "customer_id"
    )
)

# Date
date_lookup = (
    dim_date
    .select(
        "date_key",
        "full_date"
    )
)

# Order status
status_lookup = (
    dim_order_status
    .select(
        "order_status_key",
        "order_status"
    )
)

# Product
product_lookup = (
    dim_product
    .select(
        "product_key",
        "product_id",
        "product_cost"
    )
)

# Payment method
payment_method_lookup = (
    dim_payment_method
    .select(
        "payment_method_key",
        "payment_method"
    )
)

# Shipper
shipper_lookup = (
    dim_shipper
    .select(
        "shipper_key",
        "shipper_id"
    )
)

print("Dimension lookups prepared successfully.")


# ============================================================
# TASK 16 — fact_order
# ============================================================
#
# GRAIN:
#   One row per order.
#
# Source:
#   orders
#
# Connections:
#   orders.customer_id
#       -> dim_customer.customer_id
#
#   orders.order_date
#       -> dim_date.full_date
#
#   orders.status
#       -> dim_order_status.order_status
#
# Employee:
#   The actual orders source does not contain employee_id,
#   therefore employee_key is NOT included.
#
# Measures:
#   total_quantity
#   gross_sales
#   discount_amount
#   net_sales
#   total_cost
#   profit
# ============================================================

print("\n\n" + "=" * 90)
print("TASK 16 — fact_order")
print("=" * 90)

print("""
GRAIN:
One row per order.
""")

# ------------------------------------------------------------
# Aggregate order details to order level.
#
# This gives fact_order useful order-level measures while
# maintaining one row per order.
# ------------------------------------------------------------

order_detail_for_order = (
    order_details.alias("od")
    .join(
        product_lookup.alias("p"),
        F.col("od.product_id") == F.col("p.product_id"),
        "left"
    )
)

if DISCOUNT_IS_PERCENT:

    order_detail_order_metrics = (
        order_detail_for_order
        .withColumn(
            "gross_line_sales",
            F.col("od.quantity") *
            F.col("od.unit_price")
        )
        .withColumn(
            "discount_amount",
            F.col("od.quantity") *
            F.col("od.unit_price") *
            (F.col("od.discount") / F.lit(100.0))
        )
        .withColumn(
            "net_line_sales",
            F.col("gross_line_sales") -
            F.col("discount_amount")
        )
        .withColumn(
            "line_cost",
            F.col("od.quantity") *
            F.col("p.product_cost")
        )
    )

else:

    order_detail_order_metrics = (
        order_detail_for_order
        .withColumn(
            "gross_line_sales",
            F.col("od.quantity") *
            F.col("od.unit_price")
        )
        .withColumn(
            "discount_amount",
            F.col("od.discount")
        )
        .withColumn(
            "net_line_sales",
            F.col("gross_line_sales") -
            F.col("discount_amount")
        )
        .withColumn(
            "line_cost",
            F.col("od.quantity") *
            F.col("p.product_cost")
        )
    )

order_metrics = (
    order_detail_order_metrics
    .groupBy(
        F.col("od.order_id").alias("order_id")
    )
    .agg(
        F.sum("od.quantity")
            .cast(LongType())
            .alias("total_quantity"),

        F.sum("gross_line_sales")
            .cast(DoubleType())
            .alias("gross_sales"),

        F.sum("discount_amount")
            .cast(DoubleType())
            .alias("discount_amount"),

        F.sum("net_line_sales")
            .cast(DoubleType())
            .alias("net_sales"),

        F.sum("line_cost")
            .cast(DoubleType())
            .alias("total_cost")
    )
    .withColumn(
        "profit",
        F.col("net_sales") -
        F.col("total_cost")
    )
)

# ------------------------------------------------------------
# Join order with dimensions.
# ------------------------------------------------------------

fact_order = (
    orders.alias("o")

    # Customer
    .join(
        customer_lookup.alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    # Date
    .join(
        date_lookup.alias("dt"),
        F.col("o.order_date") ==
        F.col("dt.full_date"),
        "left"
    )

    # Status
    .join(
        status_lookup.alias("s"),
        F.col("o.status") ==
        F.col("s.order_status"),
        "left"
    )

    # Order-level measures
    .join(
        order_metrics.alias("m"),
        F.col("o.order_id") ==
        F.col("m.order_id"),
        "left"
    )

    .select(

        # Original order identifier
        F.col("o.order_id").alias("order_id"),

        # Dimension surrogate keys
        F.col("c.customer_key").alias("customer_key"),
        F.col("dt.date_key").alias("date_key"),
        F.col("s.order_status_key").alias("order_status_key"),

        # Original useful business attributes
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.order_date").alias("order_date"),
        F.col("o.status").alias("order_status"),

        # Measures
        F.coalesce(
            F.col("m.total_quantity"),
            F.lit(0)
        ).alias("total_quantity"),

        F.coalesce(
            F.col("m.gross_sales"),
            F.lit(0.0)
        ).alias("gross_sales"),

        F.coalesce(
            F.col("m.discount_amount"),
            F.lit(0.0)
        ).alias("discount_amount"),

        F.coalesce(
            F.col("m.net_sales"),
            F.lit(0.0)
        ).alias("net_sales"),

        F.coalesce(
            F.col("m.total_cost"),
            F.lit(0.0)
        ).alias("total_cost"),

        F.coalesce(
            F.col("m.profit"),
            F.lit(0.0)
        ).alias("profit")
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nFact order schema:")
fact_order.printSchema()

print("\nSample:")
fact_order.show(10, truncate=False)

print(f"\nfact_order rows: {fact_order.count():,}")

print("\nChecking one row per order:")

order_duplicates = (
    fact_order
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

order_duplicate_count = order_duplicates.count()

print(
    f"Duplicate order IDs: {order_duplicate_count}"
)

if order_duplicate_count == 0:
    print("✓ Grain validation passed: one row per order.")
else:
    print("WARNING: fact_order contains duplicate order IDs.")
    order_duplicates.show(20)

fact_order = write_fact(
    fact_order,
    "fact_order"
)


# ============================================================
# TASK 17 — fact_order_detail
# ============================================================
#
# GRAIN:
#   One row per order-detail record / product line.
#
# Sources:
#   orders
#   order_details
#   products
#   customers
#
# Required:
#   order_id
#   product_key
#   customer_key
#   date_key
#   quantity
#   unit_price
#   discount
#
# Calculated:
#   gross_sales
#   discount_amount
#   net_sales
#   product_cost
#   profit
# ============================================================

print("\n\n" + "=" * 90)
print("TASK 17 — fact_order_detail")
print("=" * 90)

print("""
GRAIN:
One row per order-detail record / product line within an order.
""")

fact_order_detail_base = (
    order_details.alias("od")

    # Connect order details to orders
    .join(
        orders.alias("o"),
        F.col("od.order_id") ==
        F.col("o.order_id"),
        "inner"
    )

    # Product dimension
    .join(
        product_lookup.alias("p"),
        F.col("od.product_id") ==
        F.col("p.product_id"),
        "left"
    )

    # Customer dimension
    .join(
        customer_lookup.alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    # Date dimension
    .join(
        date_lookup.alias("dt"),
        F.col("o.order_date") ==
        F.col("dt.full_date"),
        "left"
    )
)

# ------------------------------------------------------------
# Calculated sales measures.
# ------------------------------------------------------------

fact_order_detail = (
    fact_order_detail_base

    .withColumn(
        "gross_sales",
        F.col("od.quantity") *
        F.col("od.unit_price")
    )

    .withColumn(
        "discount_amount",
        F.when(
            F.lit(DISCOUNT_IS_PERCENT),
            F.col("od.quantity") *
            F.col("od.unit_price") *
            (F.col("od.discount") / F.lit(100.0))
        ).otherwise(
            F.col("od.discount")
        )
    )

    .withColumn(
        "net_sales",
        F.col("gross_sales") -
        F.col("discount_amount")
    )

    .withColumn(
        "product_cost",
        F.col("od.quantity") *
        F.col("p.product_cost")
    )

    .withColumn(
        "profit",
        F.col("net_sales") -
        F.col("product_cost")
    )

    .select(

        # Original identifiers
        F.col("od.order_detail_id").alias("order_detail_id"),
        F.col("od.order_id").alias("order_id"),
        F.col("od.product_id").alias("product_id"),

        # Dimension keys
        F.col("p.product_key").alias("product_key"),
        F.col("c.customer_key").alias("customer_key"),
        F.col("dt.date_key").alias("date_key"),

        # Business attributes
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.order_date").alias("order_date"),

        # Required measures
        F.col("od.quantity").alias("quantity"),
        F.col("od.unit_price").alias("unit_price"),
        F.col("od.discount").alias("discount"),

        # Calculated measures
        F.col("gross_sales"),
        F.col("discount_amount"),
        F.col("net_sales"),
        F.col("product_cost"),
        F.col("profit")
    )
)

print("\nFact order detail schema:")
fact_order_detail.printSchema()

print("\nSample:")
fact_order_detail.show(10, truncate=False)

print(
    f"\nfact_order_detail rows: "
    f"{fact_order_detail.count():,}"
)

# ------------------------------------------------------------
# Grain validation.
# ------------------------------------------------------------

order_detail_duplicates = (
    fact_order_detail
    .groupBy("order_detail_id")
    .count()
    .filter(F.col("count") > 1)
)

order_detail_duplicate_count = (
    order_detail_duplicates.count()
)

print(
    f"Duplicate order_detail IDs: "
    f"{order_detail_duplicate_count}"
)

if order_detail_duplicate_count == 0:
    print(
        "✓ Grain validation passed: "
        "one row per order-detail record."
    )
else:
    print(
        "WARNING: Duplicate order_detail IDs found."
    )

fact_order_detail = write_fact(
    fact_order_detail,
    "fact_order_detail"
)


# ============================================================
# TASK 18 — FACT PAYMENT
# Grain: One row per payment transaction
# Source: payments
#
# Dimensions connected:
#   - dim_customer
#   - dim_date
#   - dim_payment_method
#
# Note:
# The actual payments source does NOT contain PaymentStatus.
# Therefore payment_status is NOT included in this fact.
# ============================================================

fact_payment = (
    payments.alias("p")

    # Connect payment -> order -> customer
    .join(
        orders.select(
            "order_id",
            "customer_id"
        ).alias("o"),
        F.col("p.order_id") == F.col("o.order_id"),
        "left"
    )

    # Customer surrogate key
    .join(
        customer_lookup.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )

    # Payment date -> date dimension
    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("dt"),
        F.col("p.payment_date") == F.col("dt.full_date"),
        "left"
    )

    # Payment method -> payment method dimension
    .join(
        payment_method_lookup.alias("pm"),
        F.col("p.payment_method") == F.col("pm.payment_method"),
        "left"
    )

    .select(
        F.col("p.payment_id").alias("payment_id"),
        F.col("p.order_id").alias("order_id"),

        # Dimension keys
        F.col("c.customer_key").alias("customer_key"),
        F.col("dt.date_key").alias("date_key"),
        F.col("pm.payment_method_key").alias("payment_method_key"),

        # Descriptive attributes
        F.col("p.payment_method").alias("payment_method"),
        F.col("p.payment_date").alias("payment_date"),

        # Measure
        F.col("p.amount").cast("double").alias("amount")
    )
)

print("===== FACT PAYMENT =====")
print("Grain: One row per payment transaction")

print("\nSchema:")
fact_payment.printSchema()

print("\nSample records:")
fact_payment.show(10, truncate=False)

print("\nRecord count:")
print(fact_payment.count())

print("\nNull surrogate-key checks:")
fact_payment.select(
    F.sum(F.col("customer_key").isNull().cast("int")).alias("null_customer_key"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_key"),
    F.sum(F.col("payment_method_key").isNull().cast("int")).alias("null_payment_method_key")
).show()

# Save
write_fact(fact_payment, "fact_payment")


# ============================================================
# TASK 19 — fact_shipment
# ============================================================
#
# GRAIN:
#   One row per shipment.
#
# Sources:
#   shipments
#   orders
#   shippers
#
# Dimensions:
#   dim_customer
#   dim_shipper
#   dim_date
#
# Measures:
#   delivery_duration_days
# ============================================================

print("\n\n" + "=" * 90)
print("TASK 19 — fact_shipment")
print("=" * 90)

print("""
GRAIN:
One row per shipment.
""")

fact_shipment = (
    shipments.alias("s")

    # Connect shipment to order
    .join(
        orders.alias("o"),
        F.col("s.order_id") ==
        F.col("o.order_id"),
        "left"
    )

    # Customer
    .join(
        customer_lookup.alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    # Shipper
    .join(
        shipper_lookup.alias("sh"),
        F.col("s.shipper_id") ==
        F.col("sh.shipper_id"),
        "left"
    )

    # Ship date
    .join(
        date_lookup.alias("dt_ship"),
        F.col("s.ship_date") ==
        F.col("dt_ship.full_date"),
        "left"
    )

    # Delivery date
    .join(
        date_lookup.alias("dt_delivery"),
        F.col("s.delivery_date") ==
        F.col("dt_delivery.full_date"),
        "left"
    )

    # Delivery duration
    .withColumn(
        "delivery_duration_days",
        F.when(
            F.col("s.ship_date").isNotNull() &
            F.col("s.delivery_date").isNotNull(),
            F.datediff(
                F.col("s.delivery_date"),
                F.col("s.ship_date")
            )
        ).otherwise(
            F.lit(None).cast(IntegerType())
        )
    )

    .select(

        # Business identifiers
        F.col("s.shipment_id").alias("shipment_id"),
        F.col("s.order_id").alias("order_id"),

        # Dimension keys
        F.col("c.customer_key").alias("customer_key"),
        F.col("sh.shipper_key").alias("shipper_key"),

        # Date keys
        F.col("dt_ship.date_key")
            .alias("ship_date_key"),

        F.col("dt_delivery.date_key")
            .alias("delivery_date_key"),

        # Original dates
        F.col("s.ship_date").alias("ship_date"),
        F.col("s.delivery_date").alias("delivery_date"),

        # Original IDs
        F.col("s.shipper_id").alias("shipper_id"),

        # Measure
        F.col("delivery_duration_days")
    )
)

print("\nFact shipment schema:")
fact_shipment.printSchema()

print("\nSample:")
fact_shipment.show(10, truncate=False)

print(
    f"\nfact_shipment rows: "
    f"{fact_shipment.count():,}"
)

# Grain validation
shipment_duplicates = (
    fact_shipment
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
)

shipment_duplicate_count = shipment_duplicates.count()

print(
    f"Duplicate shipment IDs: "
    f"{shipment_duplicate_count}"
)

if shipment_duplicate_count == 0:
    print(
        "✓ Grain validation passed: "
        "one row per shipment."
    )

# Delivery-duration validation
negative_delivery_duration = (
    fact_shipment
    .filter(
        F.col("delivery_duration_days") < 0
    )
    .count()
)

print(
    f"Negative delivery durations: "
    f"{negative_delivery_duration}"
)

fact_shipment = write_fact(
    fact_shipment,
    "fact_shipment"
)


# ============================================================
# TASK 20 — fact_customer_sales
# ============================================================
#
# GRAIN:
#   One customer per day.
#
# Source:
#   fact_order_detail
#
# Measures:
#   order_count
#   quantity
#   sales
#   profit
#
# Using fact_order_detail gives us line-level sales and cost
# information while grouping it into the requested customer/day
# grain.
# ============================================================

print("\n\n" + "=" * 90)
print("TASK 20 — fact_customer_sales")
print("=" * 90)

print("""
GRAIN:
One customer per day.
""")

fact_customer_sales = (
    fact_order_detail
    .groupBy(
        "customer_key",
        "customer_id",
        "date_key",
        "order_date"
    )
    .agg(

        # Count distinct orders, not order-detail rows.
        F.countDistinct("order_id")
            .alias("order_count"),

        # Total quantity sold.
        F.sum("quantity")
            .cast(LongType())
            .alias("quantity"),

        # Net sales after discount.
        F.sum("net_sales")
            .cast(DoubleType())
            .alias("sales"),

        # Profit = net sales - product cost.
        F.sum("profit")
            .cast(DoubleType())
            .alias("profit")
    )
)

print("\nCustomer sales schema:")
fact_customer_sales.printSchema()

print("\nSample:")
fact_customer_sales.show(10, truncate=False)

print(
    f"\nfact_customer_sales rows: "
    f"{fact_customer_sales.count():,}"
)

# Grain validation
customer_sales_duplicates = (
    fact_customer_sales
    .groupBy(
        "customer_key",
        "date_key"
    )
    .count()
    .filter(F.col("count") > 1)
)

customer_sales_duplicate_count = (
    customer_sales_duplicates.count()
)

print(
    f"Duplicate customer/date combinations: "
    f"{customer_sales_duplicate_count}"
)

if customer_sales_duplicate_count == 0:
    print(
        "✓ Grain validation passed: "
        "one customer per day."
    )

fact_customer_sales = write_fact(
    fact_customer_sales,
    "fact_customer_sales"
)


# ============================================================
# TASK 21 — fact_product_sales
# ============================================================
#
# GRAIN:
#   One product per day.
#
# Source:
#   fact_order_detail
#
# Measures:
#   order_count
#   quantity_sold
#   gross_sales
#   discount
#   net_sales
#   cost
#   profit
# ============================================================

print("\n\n" + "=" * 90)
print("TASK 21 — fact_product_sales")
print("=" * 90)

print("""
GRAIN:
One product per day.
""")

fact_product_sales = (
    fact_order_detail
    .groupBy(
        "product_key",
        "product_id",
        "date_key",
        "order_date"
    )
    .agg(

        # Number of unique orders containing the product.
        F.countDistinct("order_id")
            .alias("order_count"),

        # Quantity sold.
        F.sum("quantity")
            .cast(LongType())
            .alias("quantity_sold"),

        # Sales before discount.
        F.sum("gross_sales")
            .cast(DoubleType())
            .alias("gross_sales"),

        # Total discount amount.
        F.sum("discount_amount")
            .cast(DoubleType())
            .alias("discount"),

        # Sales after discount.
        F.sum("net_sales")
            .cast(DoubleType())
            .alias("net_sales"),

        # Total product cost.
        F.sum("product_cost")
            .cast(DoubleType())
            .alias("cost"),

        # Profit.
        F.sum("profit")
            .cast(DoubleType())
            .alias("profit")
    )
)

print("\nProduct sales schema:")
fact_product_sales.printSchema()

print("\nSample:")
fact_product_sales.show(10, truncate=False)

print(
    f"\nfact_product_sales rows: "
    f"{fact_product_sales.count():,}"
)

# Grain validation
product_sales_duplicates = (
    fact_product_sales
    .groupBy(
        "product_key",
        "date_key"
    )
    .count()
    .filter(F.col("count") > 1)
)

product_sales_duplicate_count = (
    product_sales_duplicates.count()
)

print(
    f"Duplicate product/date combinations: "
    f"{product_sales_duplicate_count}"
)

if product_sales_duplicate_count == 0:
    print(
        "✓ Grain validation passed: "
        "one product per day."
    )

fact_product_sales = write_fact(
    fact_product_sales,
    "fact_product_sales"
)


# ============================================================
# FINAL FACT-TABLE SUMMARY
# ============================================================

print("\n\n" + "=" * 90)
print("PART 5 — FINAL FACT TABLE SUMMARY")
print("=" * 90)

fact_dfs = {
    "fact_order": fact_order,
    "fact_order_detail": fact_order_detail,
    "fact_payment": fact_payment,
    "fact_shipment": fact_shipment,
    "fact_customer_sales": fact_customer_sales,
    "fact_product_sales": fact_product_sales
}

fact_summary = []

for fact_name, fact_df in fact_dfs.items():

    fact_summary.append(
        (
            fact_name,
            fact_df.count(),
            len(fact_df.columns)
        )
    )

fact_summary_df = spark.createDataFrame(
    fact_summary,
    [
        "fact_name",
        "row_count",
        "column_count"
    ]
)

fact_summary_df = fact_summary_df.orderBy(
    "fact_name"
)

fact_summary_df.show(
    100,
    truncate=False
)


# ============================================================
# GRAIN DOCUMENTATION
# ============================================================

print("\n" + "=" * 90)
print("FACT TABLE GRAIN DOCUMENTATION")
print("=" * 90)

grain_documentation = [
    (
        "fact_order",
        "One row per order"
    ),
    (
        "fact_order_detail",
        "One row per order-detail/product line"
    ),
    (
        "fact_payment",
        "One row per payment transaction"
    ),
    (
        "fact_shipment",
        "One row per shipment"
    ),
    (
        "fact_customer_sales",
        "One row per customer per day"
    ),
    (
        "fact_product_sales",
        "One row per product per day"
    )
]

grain_df = spark.createDataFrame(
    grain_documentation,
    [
        "fact_table",
        "grain"
    ]
)

grain_df.show(
    truncate=False
)


# ============================================================
# OUTPUT LOCATIONS
# ============================================================

print("\n" + "=" * 90)
print("FACT TABLE OUTPUT LOCATIONS")
print("=" * 90)

for fact_name in fact_dfs.keys():
    print(
        f"{fact_name}: "
        f"{PROCESSED_PATH}/{fact_name}"
    )

print("\n" + "=" * 90)
print("PART 5 COMPLETED SUCCESSFULLY")
print("=" * 90)

PART 5 — BUILD FACT TABLES

Source DataFrames:
customers       : 10,000
orders          : 50,000
order_details   : 100,000
products        : 1,000
payments        : 45,000
shipments       : 40,000
shippers        : 10

PREPARING DIMENSION LOOKUPS
Dimension lookups prepared successfully.


TASK 16 — fact_order

GRAIN:
One row per order.


Fact order schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_status_key: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)
 |-- total_quantity: long (nullable = false)
 |-- gross_sales: double (nullable = false)
 |-- discount_amount: double (nullable = false)
 |-- net_sales: double (nullable = false)
 |-- total_cost: double (nullable = false)
 |-- profit: double (nullable = false)


Sample:
+--------+------------+--------+----------------+-----------+-----

In [14]:
# ============================================================
# PART 6 — ORDERS_JSON PROCESSING
# TASK 22
#
# Purpose:
#   Process incremental order JSON files generated by the
#   ordering API.
#
# JSON structure:
#
# {
#   "metadata": {...},
#   "order": {...},
#   "order_details": [...],
#   "payment": {...},
#   "shipment": {...},
#   "summary": {...}
# }
#
# Requirements covered:
#   1. Read orders_json using PySpark
#   2. Infer / define JSON schema
#   3. Parse nested structures
#   4. Explode nested order_details
#   5. Validate JSON records
#   6. Standardize fields
#   7. Remove duplicate orders
#   8. Identify new orders
#   9. Transform JSON into the same structure as historical data
#
# IMPORTANT:
#   We DO NOT physically concatenate the JSON files.
#   Spark reads all JSON files under the S3 prefix as one
#   logical DataFrame.
# ============================================================


from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    ArrayType
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

API_ORDERS_PATH = "s3://aws-ecommerce-s3/ecommerce/api_raw/orders"

PROCESSED_PATH = "s3://aws-ecommerce-s3/ecommerce/processed"

print("=" * 80)
print("PART 6 — TASK 22: ORDERS_JSON PROCESSING")
print("=" * 80)

print(f"\nInput JSON path:")
print(API_ORDERS_PATH)


# ============================================================
# 2. DEFINE JSON SCHEMA
# ============================================================
#
# Defining the schema explicitly is safer than allowing Spark
# to infer a different type if future JSON files contain
# unexpected/null values.
# ============================================================


order_schema = StructType([
    StructField("OrderID", IntegerType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True)
])


order_detail_schema = StructType([
    StructField("OrderDetailID", IntegerType(), True),
    StructField("OrderID", IntegerType(), True),
    StructField("ProductID", IntegerType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Discount", DoubleType(), True)
])


payment_schema = StructType([
    StructField("PaymentID", IntegerType(), True),
    StructField("OrderID", IntegerType(), True),
    StructField("PaymentMethod", StringType(), True),
    StructField("PaymentDate", StringType(), True),
    StructField("Amount", DoubleType(), True)
])


shipment_schema = StructType([
    StructField("ShipmentID", IntegerType(), True),
    StructField("OrderID", IntegerType(), True),
    StructField("ShipperID", IntegerType(), True),
    StructField("ShipDate", StringType(), True),
    StructField("DeliveryDate", StringType(), True)
])


metadata_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("source", StringType(), True)
])


# ============================================================
# 3. READ JSON FILES
# ============================================================
#
# Spark reads every JSON file under the prefix.
# No manual concatenation is required.
# ============================================================


orders_json_raw = (
    spark.read
    .schema(
        StructType([
            StructField("metadata", metadata_schema, True),
            StructField("order", order_schema, True),
            StructField(
                "order_details",
                ArrayType(order_detail_schema),
                True
            ),
            StructField("payment", payment_schema, True),
            StructField("shipment", shipment_schema, True)
        ])
    )
    .json(API_ORDERS_PATH)
)


print("\n" + "=" * 80)
print("RAW JSON DATASET")
print("=" * 80)

print("\nSchema:")
orders_json_raw.printSchema()

print("\nNumber of JSON records:")
print(orders_json_raw.count())

print("\nSample JSON records:")
orders_json_raw.show(5, truncate=False)


# ============================================================
# 4. PARSE THE NESTED ORDER STRUCTURE
# ============================================================
#
# Extract the nested "order" object into a relational structure.
# ============================================================


json_orders = (
    orders_json_raw
    .select(
        F.col("metadata.event_id").alias("event_id"),
        F.col("metadata.event_type").alias("event_type"),
        F.col("metadata.event_timestamp").alias("event_timestamp"),
        F.col("metadata.source").alias("source"),

        F.col("order.OrderID").alias("order_id"),
        F.col("order.CustomerID").alias("customer_id"),
        F.col("order.OrderDate").alias("order_date"),
        F.col("order.Status").alias("status"),

        F.col("order_details"),
        F.col("payment"),
        F.col("shipment")
    )
)


# ============================================================
# 5. STANDARDIZE ORDER FIELDS
# ============================================================


json_orders = (
    json_orders

    # Trim string fields
    .withColumn(
        "event_id",
        F.trim(F.col("event_id"))
    )
    .withColumn(
        "event_type",
        F.upper(F.trim(F.col("event_type")))
    )
    .withColumn(
        "source",
        F.lower(F.trim(F.col("source")))
    )
    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )

    # Convert date
    .withColumn(
        "order_date",
        F.to_date(F.col("order_date"), "yyyy-MM-dd")
    )

    # Convert event timestamp
    .withColumn(
        "event_timestamp",
        F.to_timestamp(F.col("event_timestamp"))
    )
)


print("\n" + "=" * 80)
print("STANDARDIZED JSON ORDERS")
print("=" * 80)

json_orders.select(
    "event_id",
    "event_type",
    "event_timestamp",
    "source",
    "order_id",
    "customer_id",
    "order_date",
    "status"
).show(10, truncate=False)


# ============================================================
# 6. VALIDATE JSON RECORDS
# ============================================================
#
# A valid order should have:
#   - event_id
#   - order_id
#   - customer_id
#   - order_date
#   - status
#   - order_details
#
# We create validation flags instead of silently deleting
# problematic records.
# ============================================================


json_orders_validated = (
    json_orders

    .withColumn(
        "invalid_missing_order_id",
        F.col("order_id").isNull()
    )

    .withColumn(
        "invalid_missing_customer_id",
        F.col("customer_id").isNull()
    )

    .withColumn(
        "invalid_missing_order_date",
        F.col("order_date").isNull()
    )

    .withColumn(
        "invalid_missing_status",
        F.col("status").isNull()
    )

    .withColumn(
        "invalid_missing_event_id",
        F.col("event_id").isNull()
    )

    .withColumn(
        "invalid_empty_details",
        (
            F.col("order_details").isNull()
            |
            (F.size(F.col("order_details")) == 0)
        )
    )

    .withColumn(
        "is_valid_order",
        ~(
            F.col("invalid_missing_order_id")
            |
            F.col("invalid_missing_customer_id")
            |
            F.col("invalid_missing_order_date")
            |
            F.col("invalid_missing_status")
            |
            F.col("invalid_missing_event_id")
        )
    )
)


print("\n" + "=" * 80)
print("JSON VALIDATION RESULTS")
print("=" * 80)

validation_summary = (
    json_orders_validated
    .agg(
        F.count("*").alias("total_json_records"),

        F.sum(
            F.col("invalid_missing_order_id").cast("int")
        ).alias("missing_order_id"),

        F.sum(
            F.col("invalid_missing_customer_id").cast("int")
        ).alias("missing_customer_id"),

        F.sum(
            F.col("invalid_missing_order_date").cast("int")
        ).alias("missing_order_date"),

        F.sum(
            F.col("invalid_missing_status").cast("int")
        ).alias("missing_status"),

        F.sum(
            F.col("invalid_missing_event_id").cast("int")
        ).alias("missing_event_id"),

        F.sum(
            F.col("invalid_empty_details").cast("int")
        ).alias("empty_order_details"),

        F.sum(
            F.col("is_valid_order").cast("int")
        ).alias("valid_orders")
    )
)

validation_summary.show(truncate=False)


# ============================================================
# 7. REMOVE DUPLICATE ORDERS
# ============================================================
#
# OrderID is the business key of the historical orders table.
#
# We keep one JSON record per OrderID.
# ============================================================


before_dedup = json_orders_validated.count()

json_orders_deduplicated = (
    json_orders_validated
    .filter(F.col("is_valid_order") == True)
    .dropDuplicates(["order_id"])
)

after_dedup = json_orders_deduplicated.count()

print("\n" + "=" * 80)
print("DUPLICATE ORDER CHECK")
print("=" * 80)

print(f"Records before duplicate removal : {before_dedup}")
print(f"Records after duplicate removal  : {after_dedup}")
print(f"Duplicate records removed        : {before_dedup - after_dedup}")


# ============================================================
# 8. IDENTIFY ORDERS ALREADY IN HISTORICAL DATA
# ============================================================
#
# Compare JSON OrderID against historical orders.
#
# Historical "orders" comes from Part 3 cleaned data.
# ============================================================


historical_order_keys = (
    orders
    .select("order_id")
    .dropDuplicates()
)


json_with_history_status = (
    json_orders_deduplicated.alias("j")
    .join(
        historical_order_keys.alias("h"),
        F.col("j.order_id") == F.col("h.order_id"),
        "left"
    )
    .withColumn(
        "is_new_order",
        F.col("h.order_id").isNull()
    )
    .drop(F.col("h.order_id"))
)


print("\n" + "=" * 80)
print("NEW ORDER IDENTIFICATION")
print("=" * 80)

new_order_summary = (
    json_with_history_status
    .agg(
        F.count("*").alias("json_orders"),
        F.sum(
            F.col("is_new_order").cast("int")
        ).alias("new_orders"),
        F.sum(
            (~F.col("is_new_order")).cast("int")
        ).alias("already_existing_orders")
    )
)

new_order_summary.show(truncate=False)


print("\nNew orders:")
(
    json_with_history_status
    .filter(F.col("is_new_order") == True)
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "status"
    )
    .orderBy("order_id")
    .show(20, truncate=False)
)


# ============================================================
# 9. CREATE THE NEW ORDERS DATASET
# ============================================================
#
# This is the JSON order data that should continue into the
# incremental pipeline.
# ============================================================


new_json_orders = (
    json_with_history_status
    .filter(F.col("is_new_order") == True)
    .select(
        "event_id",
        "event_type",
        "event_timestamp",
        "source",
        "order_id",
        "customer_id",
        "order_date",
        "status",
        "order_details",
        "payment",
        "shipment"
    )
)


print("\nNew JSON orders count:")
print(new_json_orders.count())


# ============================================================
# 10. EXPLODE NESTED ORDER DETAILS
# ============================================================
#
# One JSON order can contain multiple order_details.
#
# Example:
#
# Order 50974
#   -> detail 102428
#   -> detail 102429
#   -> detail 102430
#
# explode() converts this nested array into one row per
# order-detail record.
# ============================================================


json_order_details = (
    new_json_orders
    .select(
        "order_id",
        F.explode_outer("order_details").alias("detail")
    )
    .select(
        "order_id",

        F.col("detail.OrderDetailID")
            .alias("order_detail_id"),

        F.col("detail.ProductID")
            .alias("product_id"),

        F.col("detail.Quantity")
            .alias("quantity"),

        F.col("detail.UnitPrice")
            .alias("unit_price"),

        F.col("detail.Discount")
            .alias("discount")
    )
)


# ============================================================
# 11. STANDARDIZE ORDER DETAILS
# ============================================================


json_order_details = (
    json_order_details

    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("double")
    )

    .withColumn(
        "discount",
        F.col("discount").cast("double")
    )
)


print("\n" + "=" * 80)
print("EXPLODED ORDER DETAILS")
print("=" * 80)

print("\nSchema:")
json_order_details.printSchema()

print("\nSample:")
json_order_details.show(10, truncate=False)

print("\nOrder-detail count:")
print(json_order_details.count())


# ============================================================
# 12. VALIDATE ORDER DETAILS
# ============================================================


json_order_details_validated = (
    json_order_details

    .withColumn(
        "invalid_product_id",
        F.col("product_id").isNull()
    )

    .withColumn(
        "invalid_quantity",
        F.col("quantity").isNull()
        | (F.col("quantity") < 0)
    )

    .withColumn(
        "invalid_unit_price",
        F.col("unit_price").isNull()
        | (F.col("unit_price") < 0)
    )

    .withColumn(
        "invalid_discount",
        F.col("discount").isNull()
        | (F.col("discount") < 0)
    )

    .withColumn(
        "is_valid_detail",
        ~(
            F.col("invalid_product_id")
            |
            F.col("invalid_quantity")
            |
            F.col("invalid_unit_price")
            |
            F.col("invalid_discount")
        )
    )
)


print("\n" + "=" * 80)
print("ORDER DETAIL VALIDATION")
print("=" * 80)

json_order_details_validated.select(
    F.count("*").alias("total_details"),
    F.sum(
        F.col("invalid_product_id").cast("int")
    ).alias("invalid_product_id"),
    F.sum(
        F.col("invalid_quantity").cast("int")
    ).alias("invalid_quantity"),
    F.sum(
        F.col("invalid_unit_price").cast("int")
    ).alias("invalid_unit_price"),
    F.sum(
        F.col("invalid_discount").cast("int")
    ).alias("invalid_discount"),
    F.sum(
        F.col("is_valid_detail").cast("int")
    ).alias("valid_details")
).show()


# ============================================================
# 13. TRANSFORM JSON ORDERS TO HISTORICAL ORDERS STRUCTURE
# ============================================================
#
# Historical orders structure:
#
# order_id
# customer_id
# order_date
# status
#
# JSON structure has exactly these business fields inside
# the nested "order" object.
#
# We keep the structure compatible with the historical table.
# ============================================================


new_orders_historical_structure = (
    new_json_orders
    .select(
        F.col("order_id").cast("int").alias("order_id"),
        F.col("customer_id").cast("int").alias("customer_id"),
        F.col("order_date").alias("order_date"),
        F.col("status").alias("status")
    )
    .dropDuplicates(["order_id"])
)


print("\n" + "=" * 80)
print("NEW ORDERS — HISTORICAL STRUCTURE")
print("=" * 80)

new_orders_historical_structure.printSchema()

new_orders_historical_structure.show(
    10,
    truncate=False
)


# ============================================================
# 14. TRANSFORM JSON ORDER DETAILS TO HISTORICAL STRUCTURE
# ============================================================
#
# Historical order_details structure:
#
# order_detail_id
# order_id
# product_id
# quantity
# unit_price
# discount
# ============================================================


new_order_details_historical_structure = (
    json_order_details_validated
    .filter(F.col("is_valid_detail") == True)
    .select(
        F.col("order_detail_id").cast("int").alias("order_detail_id"),
        F.col("order_id").cast("int").alias("order_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("unit_price").cast("double").alias("unit_price"),
        F.col("discount").cast("double").alias("discount")
    )
    .dropDuplicates(["order_detail_id"])
)


print("\n" + "=" * 80)
print("NEW ORDER DETAILS — HISTORICAL STRUCTURE")
print("=" * 80)

new_order_details_historical_structure.printSchema()

new_order_details_historical_structure.show(
    10,
    truncate=False
)


# ============================================================
# 15. EXTRACT PAYMENT DATA
# ============================================================
#
# The payment is nested inside the JSON document.
# ============================================================


new_payments = (
    new_json_orders
    .select(
        F.col("payment.PaymentID").cast("int").alias("payment_id"),
        F.col("payment.OrderID").cast("int").alias("order_id"),
        F.trim(
            F.col("payment.PaymentMethod")
        ).alias("payment_method"),
        F.to_date(
            F.col("payment.PaymentDate"),
            "yyyy-MM-dd"
        ).alias("payment_date"),
        F.col("payment.Amount").cast("double").alias("amount")
    )
    .dropDuplicates(["payment_id"])
)


print("\n" + "=" * 80)
print("NEW PAYMENTS")
print("=" * 80)

new_payments.printSchema()
new_payments.show(10, truncate=False)


# ============================================================
# 16. EXTRACT SHIPMENT DATA
# ============================================================


new_shipments = (
    new_json_orders
    .select(
        F.col("shipment.ShipmentID")
            .cast("int")
            .alias("shipment_id"),

        F.col("shipment.OrderID")
            .cast("int")
            .alias("order_id"),

        F.col("shipment.ShipperID")
            .cast("int")
            .alias("shipper_id"),

        F.to_date(
            F.col("shipment.ShipDate"),
            "yyyy-MM-dd"
        ).alias("ship_date"),

        F.to_date(
            F.col("shipment.DeliveryDate"),
            "yyyy-MM-dd"
        ).alias("delivery_date")
    )
    .dropDuplicates(["shipment_id"])
)


print("\n" + "=" * 80)
print("NEW SHIPMENTS")
print("=" * 80)

new_shipments.printSchema()
new_shipments.show(10, truncate=False)


# ============================================================
# 17. FINAL VALIDATION — REFERENTIAL INTEGRITY
# ============================================================
#
# Check whether the JSON references customers/products that
# actually exist in the historical dimensions.
#
# This is important because the API may generate IDs that do
# not exist in the historical dataset.
# ============================================================


historical_customers = (
    cleaned_dfs["customers"]
    .select("customer_id")
    .dropDuplicates()
)

historical_products = (
    cleaned_dfs["products"]
    .select("product_id")
    .dropDuplicates()
)


customer_reference_check = (
    new_orders_historical_structure.alias("j")
    .join(
        historical_customers.alias("c"),
        F.col("j.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .withColumn(
        "customer_exists",
        F.col("c.customer_id").isNotNull()
    )
)


product_reference_check = (
    new_order_details_historical_structure.alias("j")
    .join(
        historical_products.alias("p"),
        F.col("j.product_id") == F.col("p.product_id"),
        "left"
    )
    .withColumn(
        "product_exists",
        F.col("p.product_id").isNotNull()
    )
)


print("\n" + "=" * 80)
print("REFERENTIAL INTEGRITY")
print("=" * 80)

print("\nCustomer reference check:")
customer_reference_check.groupBy(
    "customer_exists"
).count().show()

print("\nProduct reference check:")
product_reference_check.groupBy(
    "product_exists"
).count().show()


# ============================================================
# 18. FINAL DATASET SUMMARY
# ============================================================


print("\n" + "=" * 80)
print("PART 6 FINAL SUMMARY")
print("=" * 80)

print(
    f"Raw JSON order documents              : "
    f"{orders_json_raw.count()}"
)

print(
    f"Valid/deduplicated JSON orders        : "
    f"{json_orders_deduplicated.count()}"
)

print(
    f"New orders                             : "
    f"{new_orders_historical_structure.count()}"
)

print(
    f"New order-detail records               : "
    f"{new_order_details_historical_structure.count()}"
)

print(
    f"New payment records                    : "
    f"{new_payments.count()}"
)

print(
    f"New shipment records                   : "
    f"{new_shipments.count()}"
)


# ============================================================
# 19. SAVE PROCESSED JSON DATA
# ============================================================
#
# We save the transformed incremental datasets as Parquet.
#
# This gives us a clean structure for Part 7 incremental
# processing and later Redshift loading.
# ============================================================


new_orders_historical_structure.write \
    .mode("overwrite") \
    .parquet(
        f"{PROCESSED_PATH}/api_orders"
    )


new_order_details_historical_structure.write \
    .mode("overwrite") \
    .parquet(
        f"{PROCESSED_PATH}/api_order_details"
    )


new_payments.write \
    .mode("overwrite") \
    .parquet(
        f"{PROCESSED_PATH}/api_payments"
    )


new_shipments.write \
    .mode("overwrite") \
    .parquet(
        f"{PROCESSED_PATH}/api_shipments"
    )


print("\n" + "=" * 80)
print("SAVED PROCESSED API DATA")
print("=" * 80)

print(f"{PROCESSED_PATH}/api_orders")
print(f"{PROCESSED_PATH}/api_order_details")
print(f"{PROCESSED_PATH}/api_payments")
print(f"{PROCESSED_PATH}/api_shipments")


# ============================================================
# 20. FINAL VERIFICATION
# ============================================================


print("\n" + "=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print("\nAPI Orders:")
spark.read.parquet(
    f"{PROCESSED_PATH}/api_orders"
).show(5, truncate=False)

print("\nAPI Order Details:")
spark.read.parquet(
    f"{PROCESSED_PATH}/api_order_details"
).show(5, truncate=False)

print("\nAPI Payments:")
spark.read.parquet(
    f"{PROCESSED_PATH}/api_payments"
).show(5, truncate=False)

print("\nAPI Shipments:")
spark.read.parquet(
    f"{PROCESSED_PATH}/api_shipments"
).show(5, truncate=False)


print("\n" + "=" * 80)
print("PART 6 — TASK 22 COMPLETED")
print("=" * 80)

PART 6 — TASK 22: ORDERS_JSON PROCESSING

Input JSON path:
s3://aws-ecommerce-s3/ecommerce/api_raw/orders

RAW JSON DATASET

Schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- OrderID: integer (nullable = true)
 |    |-- CustomerID: integer (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- OrderDetailID: integer (nullable = true)
 |    |    |-- OrderID: integer (nullable = true)
 |    |    |-- ProductID: integer (nullable = true)
 |    |    |-- Quantity: integer (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |    |    |-- Discount: double (nullable = true)
 |-- payment: 

In [15]:
# ============================================================
# PART 7 — INCREMENTAL PROCESSING
# TASK 23
#
# Pipeline:
#
#   New API Orders
#          ↓
#      orders_json
#          ↓
#          S3
#          ↓
#       PySpark
#          ↓
#       Transform
#          ↓
#       Incremental Facts
#          ↓
#       Redshift
#
# Main objectives:
#   1. Do NOT reload all historical data
#   2. Identify new orders
#   3. Prevent duplicate orders
#   4. Process only new/changed records
#   5. Update appropriate fact tables
#
# IMPORTANT:
#   Historical dimensions/facts are NOT rebuilt here.
#
#   Only the new API batch is transformed.
#
#   Redshift should receive this incremental batch through
#   staging tables and MERGE/UPSERT operations.
# ============================================================


from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

API_ORDERS_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/api_raw/orders"

CHECKPOINT_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/api_raw/checkpoint/last_order_id.txt"

INCREMENTAL_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/processed/incremental"

API_ORDERS_PROCESSED_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/processed/api_orders"

API_DETAILS_PROCESSED_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/processed/api_order_details"

API_PAYMENTS_PROCESSED_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/processed/api_payments"

API_SHIPMENTS_PROCESSED_PATH = \
    "s3://aws-ecommerce-s3/ecommerce/processed/api_shipments"


print("=" * 90)
print("PART 7 — TASK 23: INCREMENTAL PROCESSING")
print("=" * 90)

print("\nAPI source:")
print(API_ORDERS_PATH)

print("\nCheckpoint:")
print(CHECKPOINT_PATH)

print("\nIncremental output:")
print(INCREMENTAL_PATH)


# ============================================================
# 2. READ THE LAST PROCESSED ORDER ID
# ============================================================
#
# The checkpoint prevents us from scanning/reprocessing the
# same new-order range every time.
#
# Example:
#
#   Last processed = 51000
#
#   API currently contains:
#
#       50001 ... 51000
#       51001 ... 52000
#
# We only need the new range:
#
#       51001 ... 52000
#
# ============================================================


import boto3

s3 = boto3.client("s3")

bucket_name = "aws-ecommerce-s3"

checkpoint_key = \
    "ecommerce/api_raw/checkpoint/last_order_id.txt"


try:

    checkpoint_object = s3.get_object(
        Bucket=bucket_name,
        Key=checkpoint_key
    )

    checkpoint_text = (
        checkpoint_object["Body"]
        .read()
        .decode("utf-8")
        .strip()
    )

    last_processed_order_id = int(checkpoint_text)

except Exception:

    # If the checkpoint does not exist yet,
    # start from zero.
    last_processed_order_id = 0


print("\n" + "=" * 90)
print("CHECKPOINT")
print("=" * 90)

print(
    f"Last processed OrderID: "
    f"{last_processed_order_id}"
)


# ============================================================
# 3. READ API JSON FILES
# ============================================================
#
# Spark reads the S3 prefix logically.
#
# We do NOT concatenate the JSON files manually.
# ============================================================


api_json = (
    spark.read
    .json(API_ORDERS_PATH)
)


print("\n" + "=" * 90)
print("API JSON SOURCE")
print("=" * 90)

print(f"JSON documents found: {api_json.count()}")

api_json.printSchema()


# ============================================================
# 4. EXTRACT THE ORDER IDENTIFIER
# ============================================================


api_orders = (
    api_json
    .select(
        F.col("metadata.event_id").alias("event_id"),
        F.col("metadata.event_type").alias("event_type"),
        F.col("metadata.event_timestamp").alias("event_timestamp"),
        F.col("metadata.source").alias("source"),

        F.col("order.OrderID").alias("order_id"),
        F.col("order.CustomerID").alias("customer_id"),
        F.col("order.OrderDate").alias("order_date"),
        F.col("order.Status").alias("status"),

        F.col("order_details"),
        F.col("payment"),
        F.col("shipment")
    )
)


# ============================================================
# 5. STANDARDIZE API ORDER DATA
# ============================================================


api_orders = (
    api_orders

    .withColumn(
        "order_id",
        F.col("order_id").cast("int")
    )

    .withColumn(
        "customer_id",
        F.col("customer_id").cast("int")
    )

    .withColumn(
        "order_date",
        F.to_date(
            F.col("order_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "status",
        F.upper(
            F.trim(
                F.col("status")
            )
        )
    )

    .withColumn(
        "event_type",
        F.upper(
            F.trim(
                F.col("event_type")
            )
        )

    )
)


# ============================================================
# 6. BASIC JSON VALIDATION
# ============================================================


api_orders = (
    api_orders

    .withColumn(
        "is_valid",
        (
            F.col("order_id").isNotNull()
            &
            F.col("customer_id").isNotNull()
            &
            F.col("order_date").isNotNull()
            &
            F.col("status").isNotNull()
        )
    )
)


invalid_api_orders = (
    api_orders
    .filter(~F.col("is_valid"))
)


print("\n" + "=" * 90)
print("JSON VALIDATION")
print("=" * 90)

print(
    f"Total API orders : {api_orders.count()}"
)

print(
    f"Invalid API orders : {invalid_api_orders.count()}"
)


# ============================================================
# 7. KEEP VALID ORDERS ONLY
# ============================================================


api_orders_valid = (
    api_orders
    .filter(F.col("is_valid"))
)


# ============================================================
# 8. REMOVE DUPLICATE ORDER EVENTS
# ============================================================
#
# OrderID is the business key.
#
# If the same order appears in multiple JSON documents,
# only one record should enter the incremental order batch.
# ============================================================


api_orders_deduplicated = (
    api_orders_valid
    .dropDuplicates(["order_id"])
)


print("\n" + "=" * 90)
print("DUPLICATE CONTROL")
print("=" * 90)

print(
    f"Before deduplication : "
    f"{api_orders_valid.count()}"
)

print(
    f"After deduplication  : "
    f"{api_orders_deduplicated.count()}"
)


# ============================================================
# 9. USE CHECKPOINT TO IDENTIFY NEW ORDERS
# ============================================================
#
# Example:
#
# checkpoint = 51000
#
# OrderID <= 51000
#       → already processed
#
# OrderID > 51000
#       → candidate for this run
#
# ============================================================


new_order_candidates = (
    api_orders_deduplicated
    .filter(
        F.col("order_id") >
        F.lit(last_processed_order_id)
    )
)


print("\n" + "=" * 90)
print("CHECKPOINT FILTER")
print("=" * 90)

print(
    f"Candidate new orders: "
    f"{new_order_candidates.count()}"
)


# ============================================================
# 10. ADD SECOND DUPLICATE-PROTECTION LAYER
# ============================================================
#
# The checkpoint is fast and handles the normal incremental
# case.
#
# But we also keep an anti-join against the already processed
# API order dataset.
#
# This makes the pipeline more idempotent.
#
# If the same order is accidentally presented again, it will
# not be inserted again.
# ============================================================


try:

    processed_api_orders = (
        spark.read
        .parquet(API_ORDERS_PROCESSED_PATH)
        .select("order_id")
        .dropDuplicates(["order_id"])
    )

    print("\nExisting processed API orders found.")

except Exception:

    processed_api_orders = None

    print(
        "\nNo existing processed API order dataset found."
    )


if processed_api_orders is not None:

    new_orders = (
        new_order_candidates.alias("new")
        .join(
            processed_api_orders.alias("old"),
            F.col("new.order_id") ==
            F.col("old.order_id"),
            "left_anti"
        )
    )

else:

    new_orders = new_order_candidates


print("\n" + "=" * 90)
print("FINAL NEW ORDER IDENTIFICATION")
print("=" * 90)

print(
    f"Orders after checkpoint : "
    f"{new_order_candidates.count()}"
)

print(
    f"Orders after anti-join  : "
    f"{new_orders.count()}"
)

print("\nNew orders:")
new_orders.select(
    "order_id",
    "customer_id",
    "order_date",
    "status"
).orderBy(
    "order_id"
).show(
    20,
    truncate=False
)


# ============================================================
# 11. HANDLE THE EMPTY-BATCH CASE
# ============================================================
#
# A production incremental pipeline must be able to run when
# there is nothing new.
#
# It should simply finish without rebuilding anything.
# ============================================================


new_order_count = new_orders.count()

if new_order_count == 0:

    print("\n" + "=" * 90)
    print("NO NEW ORDERS")
    print("=" * 90)

    print(
        "No new API orders were found."
    )

    print(
        "Historical data will NOT be reloaded."
    )

else:

    print(
        f"\nProcessing {new_order_count} new orders."
    )


# ============================================================
# 12. TRANSFORM NEW ORDERS TO HISTORICAL STRUCTURE
# ============================================================


incremental_orders = (
    new_orders
    .select(
        F.col("order_id").cast("int"),
        F.col("customer_id").cast("int"),
        F.col("order_date"),
        F.col("status")
    )
    .dropDuplicates(["order_id"])
)


# ============================================================
# 13. EXPLODE ORDER DETAILS
# ============================================================


incremental_order_details = (
    new_orders
    .select(
        "order_id",
        F.explode_outer(
            "order_details"
        ).alias("detail")
    )
    .select(

        "order_id",

        F.col(
            "detail.OrderDetailID"
        ).cast("int").alias(
            "order_detail_id"
        ),

        F.col(
            "detail.ProductID"
        ).cast("int").alias(
            "product_id"
        ),

        F.col(
            "detail.Quantity"
        ).cast("int").alias(
            "quantity"
        ),

        F.col(
            "detail.UnitPrice"
        ).cast("double").alias(
            "unit_price"
        ),

        F.col(
            "detail.Discount"
        ).cast("double").alias(
            "discount"
        )
    )
    .filter(
        F.col("order_detail_id").isNotNull()
    )
    .dropDuplicates(
        ["order_detail_id"]
    )
)


# ============================================================
# 14. TRANSFORM PAYMENTS
# ============================================================


incremental_payments = (
    new_orders
    .select(

        F.col(
            "payment.PaymentID"
        ).cast("int").alias(
            "payment_id"
        ),

        F.col(
            "payment.OrderID"
        ).cast("int").alias(
            "order_id"
        ),

        F.trim(
            F.col(
                "payment.PaymentMethod"
            )
        ).alias(
            "payment_method"
        ),

        F.to_date(
            F.col(
                "payment.PaymentDate"
            ),
            "yyyy-MM-dd"
        ).alias(
            "payment_date"
        ),

        F.col(
            "payment.Amount"
        ).cast("double").alias(
            "amount"
        )
    )
    .filter(
        F.col("payment_id").isNotNull()
    )
    .dropDuplicates(
        ["payment_id"]
    )
)


# ============================================================
# 15. TRANSFORM SHIPMENTS
# ============================================================


incremental_shipments = (
    new_orders
    .select(

        F.col(
            "shipment.ShipmentID"
        ).cast("int").alias(
            "shipment_id"
        ),

        F.col(
            "shipment.OrderID"
        ).cast("int").alias(
            "order_id"
        ),

        F.col(
            "shipment.ShipperID"
        ).cast("int").alias(
            "shipper_id"
        ),

        F.to_date(
            F.col(
                "shipment.ShipDate"
            ),
            "yyyy-MM-dd"
        ).alias(
            "ship_date"
        ),

        F.to_date(
            F.col(
                "shipment.DeliveryDate"
            ),
            "yyyy-MM-dd"
        ).alias(
            "delivery_date"
        )
    )
    .filter(
        F.col("shipment_id").isNotNull()
    )
    .dropDuplicates(
        ["shipment_id"]
    )
)


# ============================================================
# 16. VALIDATE INCREMENTAL ORDER DETAILS
# ============================================================


incremental_order_details = (
    incremental_order_details

    .filter(
        F.col("product_id").isNotNull()
    )

    .filter(
        F.col("quantity").isNotNull()
        &
        (F.col("quantity") >= 0)
    )

    .filter(
        F.col("unit_price").isNotNull()
        &
        (F.col("unit_price") >= 0)
    )

    .filter(
        F.col("discount").isNotNull()
        &
        (F.col("discount") >= 0)
    )
)


# ============================================================
# 17. BUILD INCREMENTAL FACT_ORDER_DETAIL
# ============================================================
#
# IMPORTANT:
# We do NOT rebuild the historical fact_order_detail.
#
# We only calculate the fact rows associated with the new
# orders.
# ============================================================


incremental_fact_order_detail = (
    incremental_order_details.alias("od")

    .join(
        incremental_orders.alias("o"),
        F.col("od.order_id") ==
        F.col("o.order_id"),
        "inner"
    )

    .join(
        dim_product.select(
            "product_key",
            "product_id",
            "product_cost"
        ).alias("p"),
        F.col("od.product_id") ==
        F.col("p.product_id"),
        "left"
    )

    .join(
        dim_customer.select(
            "customer_key",
            "customer_id"
        ).alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("dt"),
        F.col("o.order_date") ==
        F.col("dt.full_date"),
        "left"
    )

    .withColumn(
        "gross_sales",
        F.col("od.quantity")
        *
        F.col("od.unit_price")
    )

    .withColumn(
        "discount_amount",
        F.col("od.quantity")
        *
        F.col("od.unit_price")
        *
        (
            F.col("od.discount") / F.lit(100.0)
        )
    )

    .withColumn(
        "net_sales",
        F.col("gross_sales")
        -
        F.col("discount_amount")
    )

    .withColumn(
        "product_cost",
        F.col("od.quantity")
        *
        F.coalesce(
            F.col("p.product_cost"),
            F.lit(0.0)
        )
    )

    .withColumn(
        "profit",
        F.col("net_sales")
        -
        F.col("product_cost")
    )

    .select(

        F.col("od.order_detail_id"),
        F.col("od.order_id"),
        F.col("od.product_id"),

        F.col("p.product_key"),
        F.col("c.customer_key"),
        F.col("dt.date_key"),

        F.col("o.customer_id"),
        F.col("o.order_date"),

        F.col("od.quantity"),
        F.col("od.unit_price"),
        F.col("od.discount"),

        "gross_sales",
        "discount_amount",
        "net_sales",
        "product_cost",
        "profit"
    )
)


# ============================================================
# 18. BUILD INCREMENTAL FACT_ORDER
# ============================================================
#
# Grain:
#   One row per new order.
#
# We aggregate only the order details belonging to the
# incremental batch.
# ============================================================


incremental_order_metrics = (
    incremental_fact_order_detail
    .groupBy("order_id")
    .agg(

        F.sum(
            "quantity"
        ).alias(
            "total_quantity"
        ),

        F.sum(
            "gross_sales"
        ).alias(
            "gross_sales"
        ),

        F.sum(
            "discount_amount"
        ).alias(
            "discount_amount"
        ),

        F.sum(
            "net_sales"
        ).alias(
            "net_sales"
        ),

        F.sum(
            "product_cost"
        ).alias(
            "total_cost"
        ),

        F.sum(
            "profit"
        ).alias(
            "profit"
        )
    )
)


incremental_fact_order = (
    incremental_orders.alias("o")

    .join(
        dim_customer.select(
            "customer_key",
            "customer_id"
        ).alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("dt"),
        F.col("o.order_date") ==
        F.col("dt.full_date"),
        "left"
    )

    .join(
        dim_order_status.select(
            "order_status_key",
            "order_status"
        ).alias("s"),
        F.col("o.status") ==
        F.col("s.order_status"),
        "left"
    )

    .join(
        incremental_order_metrics.alias("m"),
        F.col("o.order_id") ==
        F.col("m.order_id"),
        "left"
    )

    .select(

        F.col("o.order_id"),
        F.col("c.customer_key"),
        F.col("dt.date_key"),
        F.col("s.order_status_key"),

        F.col("o.customer_id"),
        F.col("o.order_date"),
        F.col("o.status"),

        F.coalesce(
            F.col("m.total_quantity"),
            F.lit(0)
        ).alias("total_quantity"),

        F.coalesce(
            F.col("m.gross_sales"),
            F.lit(0.0)
        ).alias("gross_sales"),

        F.coalesce(
            F.col("m.discount_amount"),
            F.lit(0.0)
        ).alias("discount_amount"),

        F.coalesce(
            F.col("m.net_sales"),
            F.lit(0.0)
        ).alias("net_sales"),

        F.coalesce(
            F.col("m.total_cost"),
            F.lit(0.0)
        ).alias("total_cost"),

        F.coalesce(
            F.col("m.profit"),
            F.lit(0.0)
        ).alias("profit")
    )
)


# ============================================================
# 19. BUILD INCREMENTAL FACT_PAYMENT
# ============================================================


incremental_fact_payment = (
    incremental_payments.alias("p")

    .join(
        incremental_orders.select(
            "order_id",
            "customer_id"
        ).alias("o"),
        F.col("p.order_id") ==
        F.col("o.order_id"),
        "left"
    )

    .join(
        dim_customer.select(
            "customer_key",
            "customer_id"
        ).alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("dt"),
        F.col("p.payment_date") ==
        F.col("dt.full_date"),
        "left"
    )

    .join(
        dim_payment_method.select(
            "payment_method_key",
            "payment_method"
        ).alias("pm"),
        F.col("p.payment_method") ==
        F.col("pm.payment_method"),
        "left"
    )

    .select(

        F.col("p.payment_id"),
        F.col("p.order_id"),

        F.col("c.customer_key"),
        F.col("dt.date_key"),
        F.col("pm.payment_method_key"),

        F.col("p.payment_method"),
        F.col("p.payment_date"),
        F.col("p.amount")
    )
)


# ============================================================
# 20. BUILD INCREMENTAL FACT_SHIPMENT
# ============================================================


incremental_fact_shipment = (
    incremental_shipments.alias("s")

    .join(
        incremental_orders.select(
            "order_id",
            "customer_id"
        ).alias("o"),
        F.col("s.order_id") ==
        F.col("o.order_id"),
        "left"
    )

    .join(
        dim_customer.select(
            "customer_key",
            "customer_id"
        ).alias("c"),
        F.col("o.customer_id") ==
        F.col("c.customer_id"),
        "left"
    )

    .join(
        dim_shipper.select(
            "shipper_key",
            "shipper_id"
        ).alias("sh"),
        F.col("s.shipper_id") ==
        F.col("sh.shipper_id"),
        "left"
    )

    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("ship_dt"),
        F.col("s.ship_date") ==
        F.col("ship_dt.full_date"),
        "left"
    )

    .join(
        dim_date.select(
            "date_key",
            "full_date"
        ).alias("delivery_dt"),
        F.col("s.delivery_date") ==
        F.col("delivery_dt.full_date"),
        "left"
    )

    .withColumn(
        "delivery_duration_days",
        F.datediff(
            F.col("s.delivery_date"),
            F.col("s.ship_date")
        )
    )

    .select(

        F.col("s.shipment_id"),
        F.col("s.order_id"),

        F.col("c.customer_key"),
        F.col("sh.shipper_key"),

        F.col("ship_dt.date_key").alias(
            "ship_date_key"
        ),

        F.col("delivery_dt.date_key").alias(
            "delivery_date_key"
        ),

        F.col("s.ship_date"),
        F.col("s.delivery_date"),
        F.col("s.shipper_id"),

        "delivery_duration_days"
    )
)


# ============================================================
# 21. BUILD INCREMENTAL CUSTOMER SALES
# ============================================================
#
# Grain:
#   One customer per day.
#
# Only the new orders participate in this aggregation.
# ============================================================


incremental_fact_customer_sales = (
    incremental_fact_order_detail

    .groupBy(
        "customer_key",
        "customer_id",
        "date_key",
        "order_date"
    )

    .agg(

        F.countDistinct(
            "order_id"
        ).alias(
            "order_count"
        ),

        F.sum(
            "quantity"
        ).alias(
            "quantity"
        ),

        F.sum(
            "net_sales"
        ).alias(
            "sales"
        ),

        F.sum(
            "profit"
        ).alias(
            "profit"
        )
    )
)


# ============================================================
# 22. BUILD INCREMENTAL PRODUCT SALES
# ============================================================
#
# Grain:
#   One product per day.
#
# Again, only the incremental batch is processed.
# ============================================================


incremental_fact_product_sales = (
    incremental_fact_order_detail

    .groupBy(
        "product_key",
        "product_id",
        "date_key",
        "order_date"
    )

    .agg(

        F.countDistinct(
            "order_id"
        ).alias(
            "order_count"
        ),

        F.sum(
            "quantity"
        ).alias(
            "quantity_sold"
        ),

        F.sum(
            "gross_sales"
        ).alias(
            "gross_sales"
        ),

        F.sum(
            "discount_amount"
        ).alias(
            "discount"
        ),

        F.sum(
            "net_sales"
        ).alias(
            "net_sales"
        ),

        F.sum(
            "product_cost"
        ).alias(
            "cost"
        ),

        F.sum(
            "profit"
        ).alias(
            "profit"
        )
    )
)


# ============================================================
# 23. SHOW INCREMENTAL FACT RESULTS
# ============================================================


print("\n" + "=" * 90)
print("INCREMENTAL FACT RESULTS")
print("=" * 90)


print("\nFACT ORDER")
print(
    f"Rows: {incremental_fact_order.count()}"
)
incremental_fact_order.show(
    10,
    truncate=False
)


print("\nFACT ORDER DETAIL")
print(
    f"Rows: {incremental_fact_order_detail.count()}"
)
incremental_fact_order_detail.show(
    10,
    truncate=False
)


print("\nFACT PAYMENT")
print(
    f"Rows: {incremental_fact_payment.count()}"
)
incremental_fact_payment.show(
    10,
    truncate=False
)


print("\nFACT SHIPMENT")
print(
    f"Rows: {incremental_fact_shipment.count()}"
)
incremental_fact_shipment.show(
    10,
    truncate=False
)


print("\nFACT CUSTOMER SALES")
print(
    f"Rows: {incremental_fact_customer_sales.count()}"
)
incremental_fact_customer_sales.show(
    10,
    truncate=False
)


print("\nFACT PRODUCT SALES")
print(
    f"Rows: {incremental_fact_product_sales.count()}"
)
incremental_fact_product_sales.show(
    10,
    truncate=False
)


# ============================================================
# 24. SAVE ONLY THE INCREMENTAL BATCH
# ============================================================
#
# These are NOT replacements for the historical fact tables.
#
# They contain ONLY records generated by this incremental run.
#
# Redshift can load these into staging tables and MERGE them
# into the final fact tables.
# ============================================================


if new_order_count > 0:

    incremental_orders.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/orders"
        )

    incremental_order_details.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/order_details"
        )

    incremental_fact_order.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_order"
        )

    incremental_fact_order_detail.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_order_detail"
        )

    incremental_fact_payment.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_payment"
        )

    incremental_fact_shipment.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_shipment"
        )

    incremental_fact_customer_sales.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_customer_sales"
        )

    incremental_fact_product_sales.write \
        .mode("append") \
        .parquet(
            f"{INCREMENTAL_PATH}/fact_product_sales"
        )


# ============================================================
# 25. UPDATE PROCESSED API ORDER DATASET
# ============================================================
#
# This dataset is used by the anti-join on future runs.
#
# We append only the newly processed orders.
# ============================================================


if new_order_count > 0:

    (
        incremental_orders
        .write
        .mode("append")
        .parquet(
            API_ORDERS_PROCESSED_PATH
        )
    )


# ============================================================
# 26. UPDATE CHECKPOINT
# ============================================================
#
# Only update the checkpoint AFTER successful transformation
# and persistence of the incremental data.
#
# This is important:
#
#       process
#          ↓
#       save data
#          ↓
#       update checkpoint
#
# NOT:
#
#       update checkpoint
#          ↓
#       process data
#
# Otherwise a failed run could permanently skip orders.
# ============================================================


if new_order_count > 0:

    new_max_order_id = (
        new_orders
        .agg(
            F.max("order_id").alias(
                "max_order_id"
            )
        )
        .collect()[0]["max_order_id"]
    )

    print(
        f"\nNew maximum processed OrderID: "
        f"{new_max_order_id}"
    )

    s3.put_object(
        Bucket=bucket_name,
        Key=checkpoint_key,
        Body=str(
            new_max_order_id
        ).encode("utf-8")
    )

    print(
        "Checkpoint successfully updated."
    )

else:

    print(
        "\nCheckpoint was not changed "
        "because there were no new orders."
    )


# ============================================================
# 27. FINAL INCREMENTAL PIPELINE SUMMARY
# ============================================================


print("\n" + "=" * 90)
print("PART 7 — TASK 23 COMPLETED")
print("=" * 90)

print(
    f"""
Incremental processing summary
------------------------------

Previous checkpoint:
    {last_processed_order_id}

New orders processed:
    {new_order_count}

Historical data reloaded:
    NO

Duplicate protection:
    YES

Checkpoint filtering:
    YES

Historical OrderID anti-join:
    YES

Incremental fact_order:
    {incremental_fact_order.count()}

Incremental fact_order_detail:
    {incremental_fact_order_detail.count()}

Incremental fact_payment:
    {incremental_fact_payment.count()}

Incremental fact_shipment:
    {incremental_fact_shipment.count()}

Incremental fact_customer_sales:
    {incremental_fact_customer_sales.count()}

Incremental fact_product_sales:
    {incremental_fact_product_sales.count()}

Redshift strategy:
    Load incremental batch into staging tables
    and MERGE/UPSERT into final fact tables.
"""
)

print("=" * 90)

PART 7 — TASK 23: INCREMENTAL PROCESSING

API source:
s3://aws-ecommerce-s3/ecommerce/api_raw/orders

Checkpoint:
s3://aws-ecommerce-s3/ecommerce/api_raw/checkpoint/last_order_id.txt

Incremental output:
s3://aws-ecommerce-s3/ecommerce/processed/incremental

CHECKPOINT
Last processed OrderID: 51500

API JSON SOURCE
JSON documents found: 1500
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- O

In [16]:
# ============================================================
# PART 8 — S3 PROCESSED LAYER
# TASK 24 — VERIFY DIMENSIONS AND FACTS
# ============================================================
#
# Purpose:
#   Verify that all transformed dimensions and fact tables created
#   in Parts 4 and 5 are stored in the S3 processed layer in
#   Parquet format.
#
# Assignment structure:
#
# processed/
# └── ecommerce/
#     ├── dim_customer/
#     ├── dim_product/
#     ├── dim_category/
#     ├── dim_department/
#     ├── dim_supplier/
#     ├── dim_employee/
#     ├── dim_shipper/
#     ├── dim_date/
#     ├── dim_payment_method/
#     ├── dim_order_status/
#     ├── fact_order/
#     ├── fact_order_detail/
#     ├── fact_payment/
#     ├── fact_shipment/
#     ├── fact_customer_sales/
#     └── fact_product_sales/
#
# NOTE:
#   The assignment PDF uses a different bucket name:
#       s3://aws-ecommerce-data-omar-2026/processed/ecommerce/
#
#   Your actual AWS bucket is:
#       aws-ecommerce-s3
#
#   Therefore, the actual processed layer used in this project is:
#       s3://aws-ecommerce-s3/ecommerce/processed/
#
# This cell ONLY verifies the existing output.
# It does not reload, transform, or overwrite the data.
# ============================================================

from pyspark.sql import functions as F
from functools import reduce

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROCESSED_PATH = "s3://aws-ecommerce-s3/ecommerce/processed"

dimensions = [
    "dim_customer",
    "dim_product",
    "dim_category",
    "dim_department",
    "dim_supplier",
    "dim_employee",
    "dim_shipper",
    "dim_date",
    "dim_payment_method",
    "dim_order_status"
]

facts = [
    "fact_order",
    "fact_order_detail",
    "fact_payment",
    "fact_shipment",
    "fact_customer_sales",
    "fact_product_sales"
]

all_tables = dimensions + facts

print("=" * 90)
print("PART 8 — TASK 24")
print("S3 PROCESSED LAYER VERIFICATION")
print("=" * 90)

print(f"\nProcessed S3 base path:")
print(PROCESSED_PATH)

print(f"\nExpected dimensions: {len(dimensions)}")
print(f"Expected facts:      {len(facts)}")
print(f"Total expected:      {len(all_tables)}")


# ------------------------------------------------------------
# 2. Verify each table
# ------------------------------------------------------------

verification_results = []

for table_name in all_tables:

    table_path = f"{PROCESSED_PATH}/{table_name}"

    print("\n" + "-" * 90)
    print(f"Checking: {table_name}")
    print(f"Path:    {table_path}")
    print("-" * 90)

    try:

        # Read the existing Parquet output
        df = spark.read.parquet(table_path)

        row_count = df.count()
        column_count = len(df.columns)

        # Check whether Spark successfully read Parquet data
        parquet_status = "YES"

        print("Status:        FOUND")
        print("Format:        PARQUET")
        print(f"Rows:          {row_count:,}")
        print(f"Columns:       {column_count}")

        print("\nSchema:")
        df.printSchema()

        print("Sample records:")
        df.show(5, truncate=False)

        verification_results.append(
            (
                table_name,
                "FOUND",
                "PARQUET",
                row_count,
                column_count,
                "PASS"
            )
        )

    except Exception as e:

        print("Status:        NOT FOUND / READ ERROR")
        print(f"Error:         {str(e)[:500]}")

        verification_results.append(
            (
                table_name,
                "MISSING / ERROR",
                "UNKNOWN",
                0,
                0,
                "FAIL"
            )
        )


# ------------------------------------------------------------
# 3. Create verification summary
# ------------------------------------------------------------

verification_schema = [
    "table_name",
    "status",
    "format",
    "row_count",
    "column_count",
    "verification"
]

task24_verification_df = spark.createDataFrame(
    verification_results,
    verification_schema
)

print("\n" + "=" * 90)
print("TASK 24 — VERIFICATION SUMMARY")
print("=" * 90)

task24_verification_df.show(
    len(all_tables),
    truncate=False
)


# ------------------------------------------------------------
# 4. Verify dimensions separately
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("DIMENSION TABLES")
print("=" * 90)

dimension_summary_df = (
    task24_verification_df
    .filter(F.col("table_name").isin(dimensions))
    .orderBy("table_name")
)

dimension_summary_df.show(
    len(dimensions),
    truncate=False
)


# ------------------------------------------------------------
# 5. Verify facts separately
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FACT TABLES")
print("=" * 90)

fact_summary_df = (
    task24_verification_df
    .filter(F.col("table_name").isin(facts))
    .orderBy("table_name")
)

fact_summary_df.show(
    len(facts),
    truncate=False
)


# ------------------------------------------------------------
# 6. Check that every required table passed
# ------------------------------------------------------------

failed_tables = (
    task24_verification_df
    .filter(F.col("verification") != "PASS")
    .select("table_name")
    .collect()
)

passed_count = (
    task24_verification_df
    .filter(F.col("verification") == "PASS")
    .count()
)

failed_count = len(failed_tables)


# ------------------------------------------------------------
# 7. Final Task 24 result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TASK 24 FINAL RESULT")
print("=" * 90)

print(f"Expected tables : {len(all_tables)}")
print(f"Verified tables : {passed_count}")
print(f"Failed tables   : {failed_count}")

if failed_count == 0:

    print("\nSUCCESS — TASK 24 COMPLETED")
    print("-" * 90)
    print("All required dimensions and facts were found.")
    print("All tables were successfully read as Parquet.")
    print(f"Processed layer: {PROCESSED_PATH}")

else:

    print("\nWARNING — TASK 24 NEEDS ATTENTION")
    print("-" * 90)
    print("The following tables could not be verified:")

    for row in failed_tables:
        print(f"  - {row['table_name']}")

print("\n" + "=" * 90)
print("END OF TASK 24")
print("=" * 90)

PART 8 — TASK 24
S3 PROCESSED LAYER VERIFICATION

Processed S3 base path:
s3://aws-ecommerce-s3/ecommerce/processed

Expected dimensions: 10
Expected facts:      6
Total expected:      16

------------------------------------------------------------------------------------------
Checking: dim_customer
Path:    s3://aws-ecommerce-s3/ecommerce/processed/dim_customer
------------------------------------------------------------------------------------------
Status:        FOUND
Format:        PARQUET
Rows:          10,000
Columns:       9

Schema:
root
 |-- customer_key: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)

Sample records:
+------------+-----------+----------+---------+----------------